# * 맥락

## 플레이어 6명 — 닉네임 · 역할 · 보유 아이템

| # | 닉네임 | 역할 | 보유 아이템 | 개수 |
|---|---|---|---|---|
| 1 | **라면왕** | 시민 | `STATEMENT_COMPARE` 진술대조권<br>`KEYWORD_EXTRACT` 키워드 뽑기 | 2 |
| 2 | **밤하늘** | 시민 | `VOTE_DOUBLE` 자기 표 두 표로 | 1 |
| 3 | **초코비** | **마피아** | `NEWSPAPER_EDIT` 조간신문 조작권<br>`BOMB_DEFENSE` 폭탄 방어권 | 2 |
| 4 | **소금빵** | 시민 | — | 0 |
| 5 | **곰돌이** | 시민 | `SILENCE` 침묵<br>`STATEMENT_COMPARE` 진술대조권<br>`KEYWORD_EXTRACT` 키워드 뽑기 | 3 |
| 6 | **딸기우유** | **마피아** | `SILENCE` 침묵 | 1 |

### ⚠️ 역할 정보는 프롬프트에 넣지 않는다

위 표의 역할은 **사람이 결과를 채점할 때만** 본다. LLM 에게 알려주면 판정이 오염된다.
칭호 부여(게임 종료 = 역할 공개 시점)만 예외다.

### 아이템 코드

| 코드 | 이름 | 보유 제한 | 사용 시점 |
|---|---|---|---|
| `STATEMENT_COMPARE` | 특정인 지목 진술대조권 | 시민만 | 토론 |
| `KEYWORD_EXTRACT` | 특정인 지목 키워드 뽑기 | 전원 | 토론 |
| `SILENCE` | 원하는 사람 말 못하게 하기 | 전원 | 토론 |
| `VOTE_DOUBLE` | 자기 표 두 표로 계산 | 시민만 | 투표 |
| `NEWSPAPER_EDIT` | 조간신문 조작권 | 마피아만 | 밤 |
| `BOMB_DEFENSE` | AI 심판의 폭탄 방어권 | 전원 | 패시브 (심판 지목 시) |

### 아이템 사용 기록 (로그에 남아 있는 것)

| 시점 | 사용자 | 아이템 | 대상 |
|---|---|---|---|
| R1 토론 `21:02:09` | 곰돌이 | `KEYWORD_EXTRACT` | 딸기우유 |
| R1 투표 `21:03:10` | 밤하늘 | `VOTE_DOUBLE` | — |
| R1 심판 `21:04:08` | 초코비 | `BOMB_DEFENSE` | (패시브) |
| R1 밤 `21:04:38` | 초코비 | `NEWSPAPER_EDIT` | — |
| R2 토론 `21:06:38` | 딸기우유 | `SILENCE` | 라면왕 |
| R3 토론 `21:11:05` | 라면왕 | `STATEMENT_COMPARE` | 초코비 |

**미사용 3개** — 곰돌이의 `STATEMENT_COMPARE`·`SILENCE`, 라면왕의 `KEYWORD_EXTRACT`
→ 테스트할 때 이 세 개로 가정 케이스를 만들 수 있다.

### 3라운드 흐름 (시민 승)

| 라운드 | 낮 | 밤 | 생존 |
|---|---|---|---|
| 1 | 투표 동률 → AI 심판이 초코비 지목 → 방어권으로 무효 | 소금빵 사망 | 2M 3C |
| 2 | 딸기우유 최다득표 → 처형 (마피아였음) | 밤하늘 사망 | 1M 2C |
| 3 | 초코비 최다득표 → 처형 | — | **0M 2C 시민 승** |


| 기능           | 모델          | 파라미터                        | 근거                                              |
|----------------|---------------|---------------------------------|---------------------------------------------------|
| 🔍 진술대조권  | gpt-5.4-mini  | max_completion_tokens 넉넉히    | "의견 변경"과 "과거 진술 왜곡"을 구분하는 추론이 핵심 |
| 🏷️ 키워드 뽑기 | gpt-4o-mini   | temperature 0, 토큰 낮게        | 단순 추출인데 토론 중 호출이라 가장 싸고 빠른 쪽    |
| ⚖️ AI 심판     | gpt-5.4-mini  | max_completion_tokens           | 스트리밍 초반 침묵이 없음이 실측으로 확인됨         |
| 📰 조간신문    | gpt-4o-mini   | temperature 0.7~0.8             | 창작. 500자면 1024토큰으로 충분                    |
| 🏆 칭호 부여   | gpt-4o-mini   | temperature 0.8~0.9             | 대사가 재미있어야 한다                             |


## 0. 공통 — 아래 모든 기능이 함께 쓰는 부분


In [83]:
# 0-1. 설정 — 앱 시작 시 1회. 모든 기능이 이 client 를 쓴다
# time 까지 여기서 import 한다. 다섯 기능이 응답 시간을 재려고 각자 import 했다.

import json, os, re, time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # 리포 루트 .env 의 GMS_KEY, GMS_BASE 를 읽는다

GMS_BASE = os.getenv("GMS_BASE", "https://gms.ssafy.io/gmsapi")

client = OpenAI(
    api_key=os.environ["GMS_KEY"],
    base_url=f"{GMS_BASE}/api.openai.com/v1",   # /v1 까지 필수. 빼면 404
)
print("base_url :", client.base_url)


base_url : https://gms.ssafy.io/gmsapi/api.openai.com/v1/


In [84]:
# 0-2. 발화 로그 — 실서비스에서는 이 셀만 Redis 조회로 바뀐다
# 다섯 기능이 같은 파일을 읽는다. 기능별로 다시 읽지 않는다.

LOG = Path("log_analysis/scenario/scenario1/발화로그-공개-STT.jsonl")


def load_utterances(path=LOG):
    """낮 공개 발화 전체. 실서비스에서는 Redis utt:{roomId}:public 이 이 자리다.

    밤 발화 혼입을 로드 지점에서 한 번 막는다 (절대 규칙 1).
    기능마다 phase != NIGHT 를 다시 두지 않아도 되게, 부탁이 아니라 여기서 끊는다.
    """
    rows = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
    assert all(u["phase"] != "NIGHT" for u in rows), "밤 발화가 공개 로그에 섞였다"
    return rows


def before(utterances, t):
    """t **이전** 발화만. t 자체는 제외한다 (< 이고 <= 가 아니다).

    컷오프를 코드로 하는 이유 (2026-07-29 진술대조 실측) ―
    프롬프트에 "21:11:05 까지" 라고 글로 적었더니 모델이 21:11:20 을 인용했다.
    그 발화가 하필 진술대조 결과를 읽어주는 대사여서, 모순을 찾은 게 아니라
    답을 읽은 상태가 됐다.

    t 자체를 빼는 것도 의도다. 컷오프 시각의 발화는 기능마다 정답을 흘린다 ―
    아이템 사용 발화는 지목 대상을 먼저 알려주고,
    AI 심판 발동 시각의 발화는 심판 결과에 대한 반응이다.
    """
    return [u for u in utterances if u["t"] < t]


# 발화가 어느 단계에서 나왔는지를 모델에게 알려준다.
# 서버가 이미 아는 정보라 비용이 0이다 (발화수집-STT §3 의 phase 필드 요청 근거).
# DAY_START 라벨을 "시작" 으로 둔 것은 진술대조 실측값을 유지하려는 것이다 ―
# 라면왕의 DAY_START 발화 3건이 TC-5 입력에 들어간다.
PHASE_KO = {
    "DAY_START":      "시작",
    "DAY_DISCUSSION": "토론",
    "DAY_VOTE":       "투표",
    "AI_JUDGMENT":    "심판",
    "FINAL_DEFENSE":  "최후변론",
    "RESULT":         "결과",
}

ALL = load_utterances()
print(f"전체 {len(ALL)}건 · {sum(len(u['text']) for u in ALL)}자")


전체 79건 · 1967자


In [85]:
# 0-3. 이벤트 로그 — 실서비스에서는 GameSnapshotProvider 조회로 바뀐다
# 조간신문·AI 심판·칭호가 같은 파일을 읽는다.
#
# 줄 순회까지만 공통이다. 필드 추출은 기능별로 남긴다 ―
# 무엇을 **읽지 않는가** 가 기능마다 다르기 때문이다.
# 조간신문과 AI 심판은 ROLE 줄을 일부러 읽지 않고 칭호만 읽는다 (절대 규칙 2).
# 공통 파서가 roles 를 돌려주면 실수로 쓰기 쉬워지고, 그러면 규칙이 무너진다.

EVENTS = Path("log_analysis/scenario/scenario1/게임이벤트.txt")


def iter_events(path=EVENTS):
    """[시각] 본문 줄을 (t, round, phase, body) 로 흘린다.

    round·phase 는 직전 PHASE 줄에서 물고 온다 ―
    VOTE_CAST·ITEM_USE·DEATH 줄에는 라운드가 적혀 있지 않다.
    body 는 줄 끝 주석(#)을 떼고 준다.
    PHASE 줄 자신도 흘린다 ― AI 심판은 그 줄의 시각이 컷오프다.
    """
    rnd, phase = None, None
    for line in path.read_text(encoding="utf-8").splitlines():
        m = re.match(r"\[(\d\d:\d\d:\d\d)\]\s+(.+)", line)
        if not m:
            continue
        t, body = m.group(1), m.group(2).split("#")[0].strip()

        p = re.match(r"PHASE R(\d+)/(\S+)", body)
        if p:
            rnd, phase = int(p.group(1)), p.group(2)
        yield t, rnd, phase, body


print("이벤트 줄", sum(1 for _ in iter_events()), "개")


이벤트 줄 64 개


In [86]:
# 0-4. LLM 출력 파싱 — 공통. 키워드·진술대조·칭호가 같은 방식으로 받는다

def strip_fence(text):
    """모델이 ```json 으로 감싸는 경우가 있어서 벗겨낸다.

    .entity() 를 쓰지 않고 프롬프트에 형식을 직접 적는 이유는 CLAUDE.md 에 있다 ―
    Spring AI 가 스키마 지시문을 프롬프트 뒤에 자동으로 덧붙여서 노트북과
    Spring 이 서로 다른 프롬프트를 보내게 된다.
    그래서 파싱은 직접 하고, 실패는 폴백으로 흡수한다.
    """
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())


# 1. 키워드 추출

### 서버가 조회하는 것

1. 지목 대상 닉네임 — 아이템 사용 이벤트 (`ITEM_USE ... KEYWORD_EXTRACT target=`)
2. 아이템 사용 시각 — 발화 컷오프 기준
3. 낮 공개 발화 — Redis `utt:{roomId}:public`

### 서버가 LLM 에 넘기는 값

1. 지목 대상 닉네임 ← 조회 1
2. 현재 시점 ← 조회 2
3. 대상의 발화만 — `u1`…`uN` 번호 · 라운드 ← 조회 3 에 조회 2 로 컷오프

### 핵심 설계 원리

1. **대상 발화만 넘긴다.** 진술대조와 같다. 남의 발화는 노이즈다.
2. **라운드를 제한하지 않는다.** 게임 처음부터 사용 시점까지 본다 — 반복을 세는 기능이라 범위가 좁으면 5개를 채울 수 없다.
3. **`phase` 를 넣지 않는다.** 진술대조와 다른 점이다. 반복 횟수에는 어느 시점인지가 필요 없고, 라운드만 있으면 '라운드별 반복' 이 보인다.
4. **번호를 붙여 `evidence` 로 되받는다.** 원문을 생성하지 않으므로 없는 발언이 만들어질 수 없다.
5. **표현이 아니라 의미로 묶게 한다.** 글자가 같은 반복만 세면 5개를 채울 수 없다. 이것이 이 프롬프트의 핵심 규칙이다.
6. **`keyword` 는 원문에 실제로 나온 말에서 고르게 한다.** "소극적 태도" 처럼 요약해서 지은 이름을 금지한다.
7. **역할 정보를 넣지 않는다.** 토론 중에 쓰는 아이템이라 유출이 즉시 게임을 망친다.
8. **`temperature=0`, `gpt-4o-mini`.** 단순 추출인데 토론 중 호출이라 가장 싸고 빠른 쪽을 쓴다.

### 알려진 결함 — "반드시 5개" 가 원인이다

태도가 드러나는 발화가 5개가 안 되는데 5개를 요구하니 이미 쓴 발화를 재사용한다. `u3` 하나가 세 키워드의 `evidence` 로 들어갔고, 프롬프트가 예시로 명시한 묶음(`시간 낭비` = `비효율적이다`)이 분리됐다. 자세한 것은 아래 "주요 로직" 절에 있다.


### 주요 로직

1. 현재시점까지 (라운드 제한 없음 — 처음부터 사용 시점까지)
2. 지목 대상의 발화만
3. 원문에 번호를 달아 llm 이 evidence 로 근거 발언을 지목
4. 표현이 아니라 **의미로 묶어** 정확히 5개
5. 어미를 사전형으로 다듬되 원문에 없는 단어는 금지

### 코드가 하는 것

| 셀 | 처리 | 왜 |
|---|---|---|
| ① | `speaker == target` · `t < now` · `phase != NIGHT` | 필터 3개가 한 함수에. 진술대조는 컷오프와 대상 지목을 셀로 나눴다 |
| ① | `u1 [R1] 발화내용` 번호 부여 | 라운드를 붙이는 이유는 '반복'을 라운드별로 볼 수 있게 하려는 것 |
| ③ | `strip_fence` + JSON 파싱, 실패 시 `None` | 파싱 실패도 결과로 본다. 폴백 판단용 |

`ALL`(공통 셀)을 입력으로 쓴다. 진술대조는 자기 셀에서 로그를 따로 읽는다.

### LLM 이 하는 것

- 의미가 같은 발언을 하나로 묶는다
- 묶음을 대표하는 `keyword` 와 `count`, 근거 `evidence` 번호를 낸다

### 프롬프트가 요구하는 것 (규칙 6개 + 핵심 규칙)

1. 반드시 5개, `count` 큰 것부터
2. 주어진 발언에 실제로 나온 말에서 고른다, 10자 이내
3. 사전형으로 다듬는다 (`억울하면` → `억울하다`). 요약 라벨(`소극적 태도`) 금지
4. 누구나 쓰는 말(`그러니까`, `저는`)은 고르지 않는다
5. 누가 마피아인지 추측하지 않는다
6. JSON 형식으로만

### 진술대조와 다른 점

| | 키워드 | 진술대조 |
|---|---|---|
| 규칙 수 | 6개 + 핵심 규칙 블록 | 3개 |
| 개수 | **"반드시 5개를 낸다"** | "두 개를 답하시오" |
| 모델 | `gpt-4o-mini` / `max_tokens=512` | `gpt-5.4-mini` / `max_completion_tokens` |
| 번호 검증 | ❌ 없음 | ✅ `by_id` 밖의 번호는 버림 |
| 임계값 | ❌ 없음 | ✅ 3건·40자 미만이면 호출 생략 |
| 원문 치환 | ❌ `picked` 를 버린다 | ✅ 번호 → 원문 |
| 회차 반복 | ❌ 1회 | ✅ `RUNS = 3` |

진술대조는 규칙을 12개에서 3개로 줄여 성적이 좋아졌다. 키워드는 아직 안 줄였다.

### 🔴 알려진 결함 — 아직 안 고쳤다

딸기우유 11건(`now` = 21:07:30) 결과를 원문과 대조한 것이다.

```
시간 낭비     count=3  ['u1', 'u3', 'u2']    ← 묶기 정확
근거 없다     count=2  ['u6', 'u3']          ← u3 재사용, 오배정
숨길 게 없다   count=1  ['u5']
억울하다      count=1  ['u4']
비효율적이다   count=1  ['u3']                 ← u3 세 번째, 시간 낭비와 같은 묶음
```

- **`u3` 하나가 세 키워드에 배정됐다.** `u3` "방식이 비효율적이라고 한 거잖아요" 에는 근거 얘기가 없는데 `근거 없다` 의 evidence 로 들어갔다. 실제 근거는 `u6` 하나뿐인데 `count=2` 로 나왔다
- **`비효율적이다` 는 `시간 낭비` 와 같은 묶음이어야 한다.** 프롬프트가 예시로 명시한 묶음인데 분리됐다
- 원인은 **5개 강제**로 보인다. 태도가 드러나는 발화가 5개가 안 되는데 5개를 요구하니 이미 쓴 발화를 재사용한다
- **R2 발화 3건(`u9`·`u10`·`u11`)이 evidence 에 하나도 안 쓰였다.** 로그 범위를 전체 라운드로 넓힌 목적이 반쯤 안 살았다

### 나중에 붙일 것

1. `by_id` + 번호 검증 — 진술대조 ②·④를 그대로 옮긴다. evidence 로 원문을 찍어주면 오배정이 눈에 보인다
2. evidence 중복 배정 검사 — 같은 번호가 여러 키워드에 들어갔는지
3. `count == len(evidence)` 검사 — 이 셋만 넣으면 위 결함이 자동으로 걸린다
4. 부분 문자열 검사 — [ai-lab](ai-lab/src/main/java/com/ssafy/mafia/ailab/analysis/keyword/KeywordExtractService.java) 에는 있는데 노트북에 없다. 다만 `시간 남비` → `시간 낭비` 교정을 버리는 문제가 미결이다 ([todo](docs/todo.md))
5. 임계값 — [TC-9](log_analysis/scenario/scenario1/테스트-체크리스트.md) 소금빵 2건·28자


In [87]:
# 키워드 뽑기 — 발화 필터 · 프롬프트 입력 렌더

def for_keyword_extract(utterances, target, now):
    """지목 대상이 처음부터 지금까지 한 발화. 라운드를 제한하지 않는다.

    밤 발화 차단은 공통 0-2 의 load_utterances 가 이미 했다.
    컷오프도 공통 before() 와 같은 기준(< now)이라 그걸 쓴다.
    """
    return [u for u in before(utterances, now) if u["speaker"] == target]

def render(utterances):
    """u1 [R1] 발화내용 — 라운드를 붙이는 이유는 '반복'을 라운드별로 볼 수 있게 하려는 것"""
    return "\n".join(
        f"u{i} [R{u['round']}] {u['text']}"
        for i, u in enumerate(utterances, start=1)
    )



In [88]:
# 키워드 뽑기 — 프롬프트

# 주의: 이 문자열에 .format() 을 쓰지 않는다. JSON 예시의 중괄호가 깨진다.
SYSTEM = """당신은 마피아 게임의 발언 기록을 분석하는 분석관이다.
지목된 플레이어가 게임 시작부터 지금까지 한 말에서, 반복적으로 등장한 키워드를 정확히 5개 뽑는다.

핵심 규칙 — 표현이 아니라 의미로 묶는다:
서로 글자가 완전히 다른 발언이라도 같은 태도나 같은 주장을 담고 있으면 하나로 묶어서 센다.
예를 들어 "시간 낭비다" / "삼분 동안 뭘 하겠나" / "방식이 비효율적이다" 는
글자가 다르지만 모두 토론 방식에 대한 불만이므로 하나의 키워드로 묶고 count 는 3이다.
글자가 같은 반복만 세면 5개를 채울 수 없다. 반드시 의미로 묶어라.

나머지 규칙:
1. 반드시 5개를 낸다. count 가 큰 것부터 순서대로 쓴다.
2. keyword 는 그 묶음을 대표하는 표현이고, 반드시 주어진 발언에 실제로 나온 말에서 고른다.
   없는 말을 새로 만들지 않는다. 10자 이내.
3. keyword 는 사전에 실릴 형태로 다듬어 쓴다. "-다" 로 끝내거나 명사로 끝낸다.
     원문 "억울하면"       → 억울하다
     원문 "근거도 없이"     → 근거 없다
     원문 "숨길 게 없어요"   → 숨길 게 없다
     원문 "시간 남비라고"    → 시간 낭비
   다만 다듬는 것은 어미까지다. 원문에 없는 단어를 새로 만들지 않는다.
   "소극적 태도", "근거 없는 주장" 처럼 발언을 요약해서 지은 이름은 쓰지 않는다.
4. 누구나 쓰는 말(그러니까, 저는, 네, 아니)은 고르지 않는다.
   그 사람의 태도나 입장이 드러나는 것을 고른다.
5. 누가 마피아인지 추측하지 않는다.
6. 아래 JSON 형식으로만 답한다. 설명이나 코드블록 표시를 붙이지 않는다.


{"keywords": [{"keyword": "시간 낭비", "count": 3, "evidence": ["u1", "u2", "u3"]}]}

count 는 그 의미가 등장한 발언 수.
evidence 는 근거가 된 발언 번호다. 반드시 주어진 번호 안에서만 쓴다."""


USER_TMPL = """분석 대상: {target}
현재 시점: {now}

발언 목록 (시간순):
{utterances}"""


In [89]:
# 키워드 뽑기 — 호출

def extract_keywords(target, now, model="gpt-4o-mini", temperature=0, show_prompt=False):
    picked = for_keyword_extract(ALL, target, now)
    user = USER_TMPL.format(target=target, now=now, utterances=render(picked))

    if show_prompt:
        print("=== SYSTEM ===\n" + SYSTEM)
        print("\n=== USER ===\n" + user + "\n")

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=512,                  # gpt-4o-mini 는 max_tokens
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": user},
        ],
    )
    raw = resp.choices[0].message.content
    try:
        parsed = json.loads(strip_fence(raw))
    except json.JSONDecodeError:
        parsed = None                    # 파싱 실패도 결과로 본다. 폴백 판단용
    return picked, raw, parsed


In [90]:
# 키워드 뽑기 — 실행

TARGET = "딸기우유"
NOW    = "21:07:30"

_, raw, parsed = extract_keywords(TARGET, NOW)   # extract_keywords 함수가 호출 부분 포함

if parsed is None:
    print(raw)                      # JSON 파싱 실패 — 원문 그대로
else:
    for k in parsed.get("keywords", []):
        print(f"{k.get('keyword'):<12} count={k.get('count')}  {k.get('evidence')}")


시간 낭비        count=3  ['u1', 'u3', 'u6']
근거 없다        count=2  ['u6', 'u4']
비효율적이다       count=2  ['u3', 'u1']
숨길 게 없다      count=1  ['u5']
시민이다         count=1  ['u4']


### 실험 기록 — 프롬프트를 바꿔가며 나온 결과

시간 낭비        count=2  ['u1', 'u3']
시민           count=2  ['u4', 'u5']
근거           count=2  ['u6', 'u9']
말하다          count=2  ['u4', 'u7']
조용히          count=1  ['u10']

시간 낭비        count=2  ['u1', 'u3']
비효율적         count=2  ['u3', 'u1']
숨길 게 없어요     count=1  ['u5']
근거도 없이       count=1  ['u6']
억울하면         count=1  ['u4']

시간 낭비        count=1  ['u1']
비효율적이다       count=1  ['u3']
근거 없다        count=1  ['u6']
억울하다         count=1  ['u4']
숨길 게 없다      count=1  ['u5']

# 2. 진술 대조

### 서버가 조회하는 것

1. 지목 대상 닉네임 — 아이템 사용 이벤트 (`ITEM_USE ... STATEMENT_COMPARE target=`)
2. 아이템 사용 시각 — 발화 컷오프 기준. 시나리오1 은 `21:11:05`
3. 낮 공개 발화 — Redis `utt:{roomId}:public`

### 서버가 LLM 에 넘기는 값

1. 지목 대상 닉네임 ← 조회 1
2. 대상의 발화만 — `u1`…`uN` 번호 · 라운드 · phase · 시각 ← 조회 3 에 조회 2 로 컷오프

조회 2 는 프롬프트에 넣지 않는다. 컷오프 기준으로만 쓴다.
발화 원문은 넘기지만 **되받지 않는다** — 모델은 번호만 답한다.

### 핵심 설계 원리

1. **대상 발화만 넘긴다.** 전원 발화를 주면 대상 8건이 남의 발화 54건에 묻혀 3회 전부 오답이었다. 8건만 주면 3회 전부 정답이고 입력이 2415 → 535 토큰으로 줄었다.
2. **컷오프를 코드로 하고 사용 시각 자체도 뺀다.** 글로 적으면 안 지킨다. `21:11:05` 는 "초코비 씨 진술 대조해볼게요" 라 지목 대상을 먼저 알려준다.
3. **`phase` 를 넣는다.** 없으면 투표 선언 "저는 딸기우유 씨입니다" 를 정체 주장으로 읽고 없는 모순을 만든다. 라면왕 오답 3/3 → 모순 없음 3/3.
4. **USER 마지막 줄을 조건형으로 쓴다.** 명령형이면 모순이 없어도 두 개를 만들어낸다. `phase` 와 조건형은 서로 다른 문제를 고치므로 하나만 넣으면 안 된다.
5. **번호로만 답받고 원문은 서버가 치환한다.** 모델이 원문을 생성하지 않으므로 없는 발언이 만들어질 수 없다.
6. **`by_id` 밖의 번호는 코드가 버린다.** 프롬프트 품질과 무관하게 보장되는 환각 방어다.
7. **짝을 하나만 요구한다.** 여러 개를 요구하면 회차마다 개수가 흔들렸다.
8. **규칙을 12개에서 3개로 줄였고 성적이 올랐다.** 모델이 스스로 하는 것(의견 변화 vs 왜곡 구분, STT 깨짐)은 뺐고, 넣어도 안 지켜진 것(적대끼리 짝짓지 말라)도 뺐다.
9. **발화가 3건·40자 미만이면 호출하지 않는다.** 호출을 안 하면 억지 모순이 생길 수 없다. 프롬프트로 "없으면 없다고 해라" 부탁하는 것과 다르다.
10. **역할 정보를 넣지 않는다.** 규칙 없이 돌렸을 때 모델이 "마피아일 가능성까지 추론해드릴게요" 라고 먼저 제안했다.


### 주요 로직

<입력>
1. 아이템 사용시점 직전까지 입력
2. 프롬프트 간소화 
3. 지목 대상의 발화만 입력
4. phase 필요 (모델이 어느 시점 발화인지 파악하면 모순을 더 정확히 판단)
반면 3.과 같이 다른 사람이 발언이 섞이는 경우에는 혼동이 커짐.

<출력>
1. 모순 발화 짝짓기를 llm이 하도록 처리
2. llm이 없는 말을 지어내지 않도록 원문에 번호를 달아 llm이 선택하고 해당 발언을 서버에서 가져옴



### 코드가 하는 것

| 셀 | 처리 | 왜 |
|---|---|---|
| ① | `t < now` — 사용 시점 **이전**만 (`now` 자체도 제외) | 글로 "몇 시까지"라고 쓰면 모델이 안 지킨다. `21:11:05` 를 뺀 건 그 발화가 "초코비 씨 진술 대조해볼게요"라 지목 대상을 먼저 알려주기 때문 |
| ① | `phase != NIGHT` | 절대 규칙 1. 부탁이 아니라 입력에서 제거 |
| ② | **지목 대상 발화만** | 전원 발화를 주면 대상 8건이 남의 발화 54건에 묻혀 오답 3/3. 오귀속·정답 유출도 함께 막힌다 |
| ② | `u1`…`uN` 번호 부여 + `by_id` | 번호와 역인덱스를 같은 순회에서 만든다. 어긋나면 ④ 검증과 ⑤ 치환이 다른 발화를 가리킨다 |
| ② | **`phase` 를 렌더에 넣는다** — `u12 [R2 투표 21:08:10]` | 없으면 투표 선언 "저는 딸기우유 씨입니다"를 정체 주장으로 읽고 없는 모순을 만든다. 라면왕 3/3 → 0/3 |
| ④ | **임계값** — 3건·40자 미만이면 호출 생략 | [TC-4](log_analysis/scenario/scenario1/테스트-체크리스트.md). 호출을 안 하면 억지 모순이 생길 수 없다. `3`·`40` 은 잠정값(T8) |
| ④ | 없는 번호 버림 | 환각 방어. 프롬프트 품질과 무관하게 보장됨 |
| ④ | 짝이 2개 미만이면 `hasFindings=false` | 모순은 두 발언 사이에서만 성립한다 |
| ④ | JSON 파싱 실패 → `None` | 폴백 판단용 |
| ⑤ | 번호 → 원문 치환 | 여기서만 원문이 등장한다. 실서비스에서 번호는 서버에만 두고 프론트에는 원문만 내린다 |

### LLM 이 하는 것

- 충돌하는 발언 **짝 하나**를 고른다 (`ids`)
- `reason` 한 문장을 쓴다

### 프롬프트 — 규칙 3개 + 조건형 한 줄

```
1. 근거는 번호로만 답한다. 발언 원문을 다시 쓰지 않는다.
2. 누가 마피아인지 추측하지 않는다. 발언 간의 일관성만 본다.
3. 아래 JSON 형식으로만 답한다.
```

**USER 마지막 줄은 반드시 조건형이어야 한다** — `모순이 있는 경우에만 서로 충돌하는 발언 번호 두 개를 답하시오.`
명령형("번호 두 개를 답하시오")이면 모순이 없어도 두 개를 만들어낸다. 사실상 "반드시 찾아라"가 된다.

**빼도 문제없던 규칙** — 모델이 안 배워도 한다:
의견 변화 vs 과거 진술 왜곡 구분 · STT 깨짐 대응 · 중복 묶음 금지.
"같은 방향 짝짓기 금지"는 다른 이유로 뺐다 — **넣어도 지켜지지 않았다.**

**규칙 2 를 남긴 이유:** 없이 돌렸을 때 모델이 답변 끝에 "초코비의 정체가 마피아일 가능성까지 추론해드릴게요"라고 먼저 제안했다. 절대 규칙 2 위반 직전이고, 작업을 범인 찾기로 이해했다는 신호다.

### phase 와 조건형은 서로 다른 문제를 고친다

라면왕(모순 없는 인물) 각 조건 3회.

| 프롬프트 | phase 없음 | phase 있음 |
|---|---|---|
| 명령형 "번호 두 개를 답하시오" | `u12` 오독 3/3 | 오답 3/3 |
| 조건형 "모순이 있는 경우에만" | `u12` 오독 3/3 | **모순 없음 3/3** ✅ |

하나만 넣으면 TC-5 가 깨진다. 초코비는 두 조건 모두 정확해서 되던 케이스가 망가지지 않았다.

### 성적 (2026-07-29, `now` = 21:11:05)

| 대상 | 발화 | 결과 | 체크리스트 |
|---|---|---|---|
| 초코비 | 8건 289자 | 모순 있음 — `u2`×`u5` 또는 `u7`×`u8` | [TC-1](log_analysis/scenario/scenario1/테스트-체크리스트.md) · [TC-2](log_analysis/scenario/scenario1/테스트-체크리스트.md) |
| 라면왕 | 15건 317자 | 모순 없음 | [TC-5](log_analysis/scenario/scenario1/테스트-체크리스트.md) |
| 소금빵 | 2건 28자 | `INSUFFICIENT` · **호출 없음** | [TC-4](log_analysis/scenario/scenario1/테스트-체크리스트.md) |

입력 535~676 토큰, 응답 1.0~2.4초.

소금빵은 코드가 막은 것이라 **프롬프트를 검증하지 않는다.** 프롬프트 성능을 말해주는 건 초코비와 라면왕 둘이다.

### 알려진 한계

- **어느 짝이 나올지는 회차마다 다르다.** 셋 다 정답이지만 재현되지 않는다. 같은 대상을 두 번 대조하면 다른 답이 나온다
- **`u5`×`u6` 을 못 잡는다** — "처음부터 믿었습니다" 하고 2분 뒤에 그에게 투표. 유효한 후보인데 짝을 하나만 받아서 안 나온다
- **생존 검사(TC-7) 없음.** 소금빵은 R1 밤 사망자라 실서비스에서는 `TARGET_NOT_ALIVE` 로 먼저 막혀야 한다. 지금은 임계값까지 흘러가 우연히 같은 결과가 된다
- **투표 내역(TC-3) 없음.** `phase` 와 같은 "서버가 아는 사실" 계열이라 함께 볼 가치가 있다

전체 실측 근거 → [진술대조 §2-3](log_analysis/prompts/진술대조.md)


In [91]:
# ① 진술대조권 — 시점 컷오프
# 화자는 가리지 않는다. 대상 지목은 ② 에서 한다.
# 공통 0-1·0-2 를 먼저 실행한다 — 로그 읽기와 컷오프가 거기 있다.

NOW  = "21:11:05"          # 라면왕이 진술대조권을 쓴 시각
LOGS = before(ALL, NOW)

# now 자체를 제외하는 것이 이 기능에서 특히 중요하다 ―
# 21:11:05 는 "초코비 씨 진술 대조해볼게요" 라, 들어가면 누가 지목당했는지를
# 모델에게 먼저 알려준다. 컷오프 근거 전체는 공통 before() 주석에 있다.
#
# 컷오프가 동작하는지는 21:11:20(결과를 읽어주는 대사)이 빠졌는지로 확인한다.
print(f"전체 {len(ALL)}건 → {NOW} 이전 {len(LOGS)}건 "
      f"(컷오프로 {len(ALL) - len(LOGS)}건 제외)")


전체 79건 → 21:11:05 이전 65건 (컷오프로 14건 제외)


In [92]:
# ② 진술대조권 — 대상 발화만 뽑아 번호 붙이기 · 렌더
# 여기서 만든 번호가 이 기능의 환각 방어 전체를 떠받친다.
# 모델은 번호만 답하고(③ 규칙 1), 원문은 서버가 붙인다(⑤).

# phase가 입력에 포함되는게 매우 중요함, 모델이 어느 시점의 발화(토론, 투표 등)인지 맥락을 파악해야 정확히 판단할 수 있음.

def number_target(utterances, target):
    """대상 발화만 남기고 u1..uN 을 붙인다. 번호는 시간순.

    rows  — 렌더용. (번호, 발화) 쌍
    by_id — 검증·치환용. "u5" → 발화 원본

    둘을 같은 순회에서 만든다. 따로 만들면 번호가 어긋나고, 그러면
    ④ 의 검증과 ⑤ 의 치환이 서로 다른 발화를 가리킨다.

    다른 사람 발화를 함께 넣지 않는 이유 (2026-07-29, 각 조건 3회 실측):
      전체 65건        → 3회 정답. 단, 곰돌이가 모순을 짚어주는 발화
                         3건(21:05:58 · 21:06:30 · 21:10:58) 덕이었다
      해설 3건만 제거   → 3회 전부 오답. u1(밤하늘 얘기) · u4(자기 변호)
                         처럼 딸기우유와 무관한 발화를 u8 과 짝지었다
      대상 발화 8건만   → 3회 전부 정답. 입력 2415 → 535 토큰
    맥락 54건은 노이즈로 작동하고, 해설이 그 노이즈를 덮고 있었을 뿐이다.

    포기한 것: 질문이 안 보이므로 "회피" 는 탐지할 수 없다.
    §7-1 이 아이템 목적에 넣어둔 항목이지만 v0 은 모순만 본다.
    """
    rows, by_id, n = [], {}, 0
    for u in utterances:
        if u["speaker"] != target:
            continue
        n += 1
        uid = f"u{n}"
        by_id[uid] = u
        rows.append((uid, u))
    return rows, by_id


# PHASE_KO 는 공통 0-2 로 옮겼다. 다섯 기능이 같은 라벨을 쓴다.

def render_sc(rows):
    """u12 [R2 투표 21:08:10] 저는 딸기우유 씨입니다

    라운드를 붙이는 이유는 '나중에 한 말' 을 모델이 알 수 있게 하려는 것이다.
    모순 판정에는 어느 쪽이 과거 진술인지가 필요하다.

    phase 를 붙이는 이유 (2026-07-29 라면왕 실측, 각 조건 3회):
      투표 단계의 "저는 딸기우유 씨입니다" 는 투표 선언인데,
      phase 가 없으면 모델이 "나는 딸기우유다" 라는 정체 주장으로 읽고
      u4 "딸기우유 씨 반응 속도가 빠르네요" 와 모순이라고 답했다.
      라면왕은 모순이 없는 대상이므로 TC-5 의 실패 조건이다.

        | 프롬프트          | phase 없음      | phase 있음        |
        | 명령형 "답하시오"  | u12 오독 3/3    | 오답 3/3          |
        | 조건형 "있는 경우" | u12 오독 3/3    | 모순 없음 3/3 ✅  |
      두 변경이 서로 다른 문제를 고친다. phase 는 오독을, 조건형(③)은
      없을 때 없다고 말하는 능력을 담당한다. 하나만 넣으면 안 된다.
      초코비는 두 조건 모두 3/3 u2×u5 로, 되던 케이스가 망가지지 않았다.

    화자 이름은 전부 대상이라 뺐다. 입력 541 → 525 토큰.
    """
    return "\n".join(
        f"{uid:>3} [R{u['round']} {PHASE_KO.get(u['phase'], u['phase'])} {u['t']}] {u['text']}"
        for uid, u in rows
    )


In [93]:
# ③ 진술대조권 — 프롬프트
# 주의: SYSTEM_SC 에 .format() 을 쓰지 않는다. JSON 예시의 중괄호가 깨진다.
#      값을 끼우는 것은 USER_SC 쪽뿐이다.
#
# 규칙이 3개뿐인 이유 — 12개짜리 버전보다 실측 성적이 좋았다 (2026-07-29).
#   뺀 것: 의견 변화 vs 과거 진술 왜곡 구분 → 안 가르쳐도 모델이 한다
#          STT 로 깨진 발화 처리          → 안 가르쳐도 모델이 한다
#          "적대끼리 짝짓지 말라"          → 넣어도 안 지켜졌다 (우호끼리 짝지음)
#   남긴 것: 아래 3개. 모델이 스스로 못 하거나, 계약이라 지켜져야 하는 것
#
# 규칙 2 를 남긴 근거: 규칙 없이 돌렸을 때 모델이 답변 끝에
#   "초코비의 정체가 마피아일 가능성까지 추론해드릴게요" 라고 먼저 제안했다.
#   절대 규칙 2 위반 직전이고, 작업을 범인 찾기로 이해했다는 신호다.
#
# ai-lab 으로 옮길 때 SYSTEM_SC 는 .st 파일이 되고, USER_SC 는 그대로 남는다.

SYSTEM_SC = """마피아 게임의 낮 발언 기록에서, 한 사람의 발언이 서로 앞뒤가 맞지 않는지 본다.

주어진 발언은 모두 분석 대상 한 사람의 것이고, 번호(u1, u2 ...)가 붙어 있다.

규칙:
1. 근거는 번호로만 답한다. 발언 원문을 다시 쓰지 않는다.
2. 누가 마피아인지 추측하지 않는다. 발언 간의 일관성만 본다.
3. 아래 JSON 형식으로만 답한다. 설명이나 코드블록 표시를 붙이지 않는다.

모순을 찾았을 때:
{"hasFindings": true, "ids": ["u2", "u5"], "reason": "한 문장"}

모순이 없을 때:
{"hasFindings": false, "ids": [], "reason": ""}

주의사항: 
1. 모순이 없는 경우도 많으므로 명백한 모순이 없는 경우 모순 없음으로 판단한다.
"""


# "번호 두 개" 로 개수를 못 박은 결과 3회 전부 같은 짝이 나왔다.
# 여러 개를 요구하면 회차마다 개수가 흔들렸던 전례가 있다(어제 8회 관찰).
# 여러 개가 필요해지면 그때 조합을 코드로 옮긴다(판정 B안).
USER_SC = """{log}

분석 대상: {target}
모순이 있는 경우에만 서로 충돌하는 발언 번호 두 개를 답하시오."""

In [94]:
# ④ 진술대조권 — 호출 + 번호 검증
# client 는 공통 셀에서 만든 것, strip_fence 도 공통 셀 것을 쓴다.
# 이 셀을 돌리기 전에 공통 셀을 먼저 실행해야 한다. time 도 공통 0-1 에서 온다.

def compare(target, utterances, model="gpt-5.4-mini",
            max_completion_tokens=2048, show_prompt=False):
    """모순 짝 하나를 받아서 검증까지 끝낸다.

    반환 4개 — (by_id, raw, parsed, info)
      by_id  — "u5" → 발화 원본. ⑤ 에서 번호를 원문으로 치환할 때 쓴다
      raw    — 모델 응답 원문. 파싱이 실패했을 때 눈으로 보려고 남긴다
      parsed — 검증까지 끝난 dict. 파싱 실패면 None
      info   — 응답 시간·토큰. 회차를 비교하려면 필요하다

    max_completion_tokens 를 넉넉히 두는 이유: gpt-5.x 는 추론 모델이라
    추론 토큰이 이 상한을 같이 먹는다. 실측에서는 출력이 57~72토큰이라
    2048 은 과하지만, 상한은 과금이 아니라 잘림 방지용이라 손해가 없다.
    """
    rows, by_id = number_target(utterances, target)

    # 임계값 — 대상 발언이 3건 미만 또는 40자 미만이면 호출하지 않는다.
    # 호출하지 않으면 억지 모순이 만들어질 일도 없다 (TC-4 소금빵 2건·28자).
    # 프롬프트로 "없으면 없다고 해라" 부탁하는 것과 다르다. 부탁은 새고
    # 호출을 안 하는 것은 안 샌다.
    # 3·40 은 잠정값이다 — R1 토론 중에 쓰면 6명 중 3명이 미달이다 (T8 미결).
    if len(by_id) < 3 or sum(len(u["text"]) for u in by_id.values()) < 40:
        return by_id, None, {"hasFindings": False, "ids": [],
                             "reason": "INSUFFICIENT"}, {"sec": 0.0}

    user = USER_SC.format(log=render_sc(rows), target=target)
    if show_prompt:
        print("=== SYSTEM ===\n" + SYSTEM_SC)
        print("\n=== USER ===\n" + user + "\n")

    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        temperature=0,          # 0 이어도 실행마다 달라진다. 그래서 ⑤ 가 여러 번 돈다
        max_completion_tokens=max_completion_tokens,   # gpt-5.x 는 max_tokens 를 주면 400
        messages=[{"role": "system", "content": SYSTEM_SC},
                  {"role": "user",   "content": user}],
    )
    info = {"sec": time.perf_counter() - t0,
            "in":  resp.usage.prompt_tokens,
            "out": resp.usage.completion_tokens}

    raw = resp.choices[0].message.content
    try:
        parsed = json.loads(strip_fence(raw))
    except json.JSONDecodeError:
        return by_id, raw, None, info      # 파싱 실패도 결과로 본다. 폴백 판단용

    # 없는 번호는 버린다 — 프롬프트 품질과 무관하게 보장되는 환각 방어.
    # 프롬프트를 어떻게 고쳐도 by_id 밖의 번호는 여기서 죽는다.
    ids = parsed.get("ids", [])
    bad = [i for i in ids if i not in by_id]
    if bad:
        print("없는 번호 버림:", bad)
    parsed["ids"] = [i for i in ids if i in by_id]
    if len(parsed["ids"]) < 2:
        parsed["hasFindings"] = False      # 짝이 안 되면 결과가 아니다

    return by_id, raw, parsed, info

In [96]:
# ⑤ 진술대조권 — 실행
# 번호를 원문으로 되돌리는 곳. 이 기능에서 발화 원문이 등장하는 유일한 지점이다.

TARGET = "초코비" # 초코비 : 모순 있음. 라면왕, 소금빵 -> 모순 없음으로 처리되어야 함.
RUNS   = 3          # 1회로 판단하지 않는다. temperature=0 이어도 실행마다 달라진다
                    # 매번 나오는 것과 그 회차에만 나온 것을 구별해야 한다

for i in range(1, RUNS + 1):
    by_id, raw, parsed, info = compare(TARGET, LOGS)

    head = f"{i}회차 — {info['sec']:.1f}초"
    if info.get("in"):                      # 임계값에 걸려 호출을 건너뛰면 토큰이 없다
        head += f" | 입력 {info['in']} · 출력 {info['out']} 토큰"
    print("=" * 70); print(head); print("=" * 70)

    if parsed is None:
        print("JSON 파싱 실패 —\n" + raw)
    elif not parsed.get("hasFindings"):
        # reason 이 INSUFFICIENT 면 "발언이 부족했다",
        # 비어 있으면 "찾아봤는데 모순이 없었다". 유저에게 보여줄 문구가 다르다
        print("모순 없음", parsed.get("reason") or "")
    else:
        print(parsed.get("reason"), parsed.get("ids"))
        for uid in parsed["ids"]:
            u = by_id[uid]      # 번호 → 원문 치환. 모델은 원문을 쓰지 않았으므로
                                # 없는 발언이 만들어질 수 없고 인용이 잘릴 일도 없다
            print(f"  {uid} [R{u['round']} {u['t']}] {u['text']}")
    print()

# 실서비스 주의 — by_id 와 번호는 서버에만 둔다.
# 프론트로는 치환된 원문만 내려보낸다 (AI-API명세 §2-5).

1회차 — 2.3초 | 입력 555 · 출력 72 토큰
u7은 딸기우유 씨를 처형하는 쪽에 섰다고 하고 u8은 딸기우유 씨를 의심한 적이 없다고 해 서로 앞뒤가 맞지 않습니다. ['u7', 'u8']
  u7 [R3 21:10:23] 저는 딸기우유 씨를 처형하는데 편성했습니다 마피아가 동료를 죽이다니요
  u8 [R3 21:10:45] 저는 딸기우유 씨를 의심한 적이 없습니다 그래서 어제 마음이 아팠어요

2회차 — 2.2초 | 입력 555 · 출력 77 토큰
u2는 딸기우유 씨를 수상하다고 하며 투표하겠다고 했고, u8은 딸기우유 씨를 의심한 적이 없다고 말해 서로 앞뒤가 맞지 않습니다. ['u2', 'u8']
  u2 [R1 21:01:12] 저는 딸기우유 씨가 제일 수상해요 아까부터 논의를 계속 막고 있어요 이번 투표는 달기우유 씨한테 너를 생각입니다
  u8 [R3 21:10:45] 저는 딸기우유 씨를 의심한 적이 없습니다 그래서 어제 마음이 아팠어요

3회차 — 1.2초 | 입력 555 · 출력 81 토큰
u2는 딸기우유 씨를 수상하다고 하면서 투표하겠다고 했고, u5는 처음부터 딸기우유 씨를 믿었다고 말해 서로 앞뒤가 맞지 않습니다. ['u2', 'u5']
  u2 [R1 21:01:12] 저는 딸기우유 씨가 제일 수상해요 아까부터 논의를 계속 막고 있어요 이번 투표는 달기우유 씨한테 너를 생각입니다
  u5 [R2 21:06:20] 마음이 바뀐 겁니다 그게 이상한가요 딸기우유 씨는 아까부터 논리가 있었어요 저는 처음부터 딸기유유 씨를 믿었습니다



# 3. 조간신문

### 서버가 조회하는 것

1. 낮 공개 발화 — Redis `utt:{roomId}:public`
2. 라운드별 처형 결과 — 처형자 · 처형된 사람의 역할 · 처형이 없었으면 그 사유
3. 라운드별 밤 사망자
4. 신문 호수 — 기반이 된 낮의 라운드 번호

### 서버가 LLM 에 넘기는 값

1. 해당 라운드의 낮 발화 (시각 · phase · 화자 · 본문) ← 조회 1 에서 `DAY_DISCUSSION` · `DAY_VOTE` · `AI_JUDGMENT` · `FINAL_DEFENSE` 만
2. 결과 한 줄 — `결과: 딸기우유 처형` 또는 `결과: 처형 없음 (방어권_발동)` ← 조회 2 에서 **역할을 뺀 것**

조회한 4개 중 2개만 넘긴다. 나머지는 서버가 아침에 조립한다.

- **처형된 사람의 역할** — 절대 규칙 2. 마피아 한 명이 확정되면 나머지를 역산하는 재료가 된다. 처형 결과만으로는 그 추론이 돌아가지 않는다
- **밤 사망자** — 밤에 조작권 소지자에게 본문을 보여줄 시점에는 아직 일어나지 않은 일이다. 서버가 아침에 맨 앞에 붙인다
- **호수·머리글** — 서버가 붙이면 모델이 틀릴 일이 없어진다

이 분리가 조작권의 사정거리를 정한다. 마피아가 손댈 수 있는 것은 토론 요약뿐이고, 처형·역할 공개·사망자 줄은 구조적으로 조작이 닿지 않는다.


## [1] 정상 케이스 기준

### 서버에서 줘야 할 것 — LLM 입력값

**1. 낮 대화 로그**

- 전날 낮 발화 전체 (토론 ~ 최후변론 또는 AI 심판 대화까지)
- 화자: **전원**

**2. 처형 결과값** — 낮이 어떻게 끝났는지에 따라

| 종료 경로 | 함께 줄 것 |
|---|---|
| AI 심판 | 1. 방어권 작동 여부, 2. 사망자 이름 (없으면 `없음`) |
| 최후변론 | 1. 사망 여부, 2. 사망자 이름 |

---

### 1. **마피아에게 전달되는 신문 예시** - 조작권 소지자(초코비)에게

【601 조간】 제2호

■ "시간 낭비"라던 딸기우유 님, 하루 만에 최다 지목자로

"느낌으로 가야죠" — 딸기우유 님의 한마디에 회의장이 얼어붙었다. 곰돌이 님이
"그건 마피아가 제일 좋아하는 방식"이라 받아쳤고, 초코비 님은 곧바로 "제일
수상하다"며 손가락을 들었다.

밤하늘 님은 라면왕 님의 매끄러운 진행을 의심했고, 말 없던 소금빵 님도 그 시선을
피하지 못했다.

투표함이 열렸으나 표가 갈렸다. AI 심판이 부른 이름은 초코비 님 — 그러나 폭탄은
터지지 않았다.

---

### 2. **전원에게 전달되는 신문 예시** - 전원에게 & 마피아의 조작 반영

【601 조간】 제2호

■ 소금빵 님이 마피아에게 살해당했습니다.
■ "시간 낭비"라던 딸기우유 님, 하루 만에 최다 지목자로

"느낌으로 가야죠" — 딸기우유 님의 한마디에 회의장이 얼어붙었다. 곰돌이 님이
"그건 마피아가 제일 좋아하는 방식"이라 받아쳤고, 회의장의 시선이 딸기우유 님에게
모였다.

밤하늘 님은 라면왕 님의 매끄러운 진행을 의심했고, 말 없던 소금빵 님도 그 시선을
피하지 못했다.

투표함이 열렸으나 표가 갈렸다. AI 심판이 부른 이름은 초코비 님 — 그러나 폭탄은
터지지 않았다.


### 설계 원리 — 코드가 하는 것

| 셀 | 처리 | 왜 |
|---|---|---|
| ① | `phase` 화이트리스트 4종 (`DAY_DISCUSSION`·`DAY_VOTE`·`AI_JUDGMENT`·`FINAL_DEFENSE`) | 절대 규칙 1을 부탁이 아니라 입력에서 막는다. `DAY_START`는 어제 신문 얘기라 신문이 자기를 인용하고, `RESULT`는 사망자 줄과 겹친다 |
| ① | 이벤트에서 `ROLE` 줄을 안 읽는다 | 그 한 줄에 전원의 역할이 있다 |
| ② | **화자 전원 · 번호 없음** | 진술대조와 반대다. 회의 전체를 요약하고, 평문 출력이라 번호를 되받을 계약이 없다 |
| ② | `말한사람 X · 발언 Y` 라벨 분리 | 한 줄에 이름이 둘이면 주체가 뒤집힌다. `21:02:00`이 2회 연속 "소금빵이 경고했다"로 나왔는데 실제로는 밤하늘이 소금빵을 지적한 말이었다 |
| ② | 결과 한 줄을 붙이되 `role`은 버린다 | 서버가 아는 사실을 안 주면 모델이 추측한다. 단 마피아 한 명이 확정되면 나머지를 역산하는 재료가 되므로 역할만 뺀다 |
| ④ | `assert phase != NIGHT` + 밤 전용 어절 검사(입력·출력 양쪽) | 유출 방어를 두 겹으로 둔다. 낮에도 쓰는 어절을 빼는 이유는 오탐이 나면 검사가 무시당하기 때문이다 |
| ④ | 검사를 호출 함수 **안**에 둔다 | TC-12는 최우선 회귀 테스트라 빼먹을 수 없게 만든다 |
| ④ | 길이는 `max_tokens`로 막는다 | 프롬프트의 글자수 지시는 지켜지지 않았다. 1토큰 ≈ 1.5자이므로 300이면 450자쯤 |
| ⑤ | 머리글·사망자 줄·역할 공개는 **서버** | LLM이 호수를 틀릴 일이 없어지고, 역할 공개 주체가 서버로 단일화된다 |
| ⑤ | 자동 검사는 500자 초과뿐 | 역할 단정·톤·인용은 표현이 계속 바뀌어 목록으로 못 잡는다. 문자열로 자동화하는 것은 밤 로그 유출뿐 |

### 2단 조립

`write_newspaper` 가 돌려주는 `body` 가 **밤에 조작권 소지자에게 보여주는 것 그대로**다. 아침에 서버가 `masthead` + `casualty_lines` 를 맨 앞에 붙인다.

- 밤에는 사망자 줄을 만들 수 없다 — 밤 살해가 아직 안 일어났다
- 그래서 **처형·역할 관련 조작이 구조적으로 무효화**된다. 조작권이 닿는 것은 토론 요약뿐이다
- 사망자 줄이 앞에 오는 이유는 17-6 — 낮 시작에 10초만 표시한다
- LLM 실패 시 사망자 줄만 나가도 정보 손실이 없다. 잃는 것은 재미뿐이다

- 사용 모델 : gpt-4o-mini
- max token : 300 (한국어 400자 정도)
- `temperature=0.8` 은 창작이라 그렇고, `info["sec"]` 를 재는 이유는 생성 창이 R1 4초 / R2 9초라 재시도 예산이 여기서 정해지기 때문이다.


### 프롬프트 설계 원리

1. **역할 명확화**  
   LLM을 단순 요약기가 아닌 **타블로이드 신문 기자**로 정의해 기사체와 편집 관점을 유도했다.

2. **선택 기준 우선**  
   "회의를 모두 기록"이 아니라 **가장 극적인 장면만 선택**하도록 지시해 요약문이 아닌 기사처럼 작성하게 했다.

3. **편집 리듬 유도**  
   인용 뒤에 반드시 반응이나 분위기 변화를 붙이도록 하여 **인용 → 반응** 구조의 기사 리듬을 만들었다.

4. **Few-shot 활용**  
   다양한 기사 예시를 제공해 문체보다 **정보 선택, 헤드라인, 문단 구성**을 자연스럽게 학습하도록 했다.

5. **성공 기준 제시**  
   "핵심 갈등이 보이는가", "신문 기사처럼 읽히는가" 등의 기준을 명시해 생성 품질을 안정화했다.

### 신문 생성 시점
최후변론 or AI심판 직후, 전환 애니메이션 동안 llm 호출을 받는 것을 목표

In [32]:
# ===== 셀 1 / 5 : 낮 컷오프 + 결과 이벤트 =====
# 화자를 가리지 않는다. 진술대조·키워드와 반대로 회의 전체를 요약한다.
# 공통 0-1 ~ 0-3 을 먼저 실행한다.

# DAY_START 와 RESULT 를 뺀 이유:
#   DAY_START ― 어제 신문에 대한 반응이라 신문이 자기 얘기를 쓰게 된다
#   RESULT    ― 처형 반응은 사망자 줄과 중복이고, 생성 창이 3초 더 줄어든다
DAY_PHASES_NP = ("DAY_DISCUSSION", "DAY_VOTE", "AI_JUDGMENT", "FINAL_DEFENSE")


def day_log(utterances, round_no):
    """R{round_no} 낮 발화 전체. 화자를 가리지 않는다.

    컷오프를 phase 화이트리스트로 하는 이유는 공통 before() 와 같다 ―
    "몇 시까지" 를 프롬프트에 글로 적으면 모델이 안 지킨다.
    """
    return [u for u in utterances
            if u["round"] == round_no and u["phase"] in DAY_PHASES_NP]


def load_events():
    """라운드별 낮 종료 결과와 밤 사망자만 뽑는다.

    조간신문에 필요한 것은 이 셋뿐이다. 투표 내역·아이템 이력은 칭호 몫이다.
    ROLE 줄(21:00:00)은 읽지 않는다 ― 전원의 역할이 적혀 있다.
    줄 순회는 공통 iter_events() 가 한다.
    """
    rounds = {}
    for _t, rnd, _phase, body in iter_events():
        if rnd is None:
            continue
        rounds.setdefault(rnd, {"execution": None, "role": None,
                                "reason": None, "night_kill": None})

        d = re.match(r"DEATH (\S+) cause=(\S+)(?: role=(\S+))?", body)
        if d:
            who, cause, role = d.groups()
            if cause == "EXECUTION":
                rounds[rnd]["execution"] = who
                rounds[rnd]["role"] = role     # LLM 입력에서는 버린다 (절대 규칙 2)
            elif cause == "NIGHT_KILL":
                rounds[rnd]["night_kill"] = who
            continue

        e = re.match(r"EXECUTION none reason=(\S+)", body)
        if e:
            rounds[rnd]["reason"] = e.group(1)
    return rounds


ROUNDS = load_events()

for r in sorted(ROUNDS):
    rows = day_log(ALL, r)
    print(f"R{r} 낮 {len(rows):2}건 {sum(len(u['text']) for u in rows):4}자 | {ROUNDS[r]}")


R1 낮 28건  780자 | {'execution': None, 'role': None, 'reason': '방어권_발동', 'night_kill': '소금빵'}
R2 낮 25건  681자 | {'execution': '딸기우유', 'role': '마피아', 'reason': None, 'night_kill': '밤하늘'}
R3 낮 19건  427자 | {'execution': '초코비', 'role': '마피아', 'reason': None, 'night_kill': None}


In [33]:
# ===== 셀 2 / 5 : 렌더 =====
# 진술대조와 달리 발화 번호와 by_id 가 없다. 평문 출력이라 번호를 되받을 계약이 없다.
# 대신 화자를 넣는다. 누가 누구를 의심했는지가 기사의 내용이다.

# PHASE 라벨은 공통 0-2 의 PHASE_KO 를 쓴다.

def result_line(rounds, round_no):
    """LLM 입력의 마지막 줄. 서버가 아는 사실을 안 주면 모델이 추측한다.

    role 은 넣지 않는다 ― 마피아 한 명이 확정되면 나머지를 추론하는 재료가 된다.
    처형 결과만으로는 그 추론이 돌아가지 않는다.
    """
    r = rounds[round_no]
    if r["execution"]:
        return f"결과: {r['execution']} 처형"
    return f"결과: 처형 없음 ({r['reason'] or '사유 없음'})"

def render_np(rows, rounds, round_no):
    """[21:02:00] 토론 · 말한사람 밤하늘 · 발언 소금빵 씨는 계속 조용한데…

    화자와 발언을 라벨로 끊는다. 2026-07-30 실측 ― 한 줄에 이름이 둘이면
    (화자 + 언급 대상) 모델이 주체를 뒤집는다. 21:02:00 이 2회 연속
    "소금빵이 경고했다" 로 나왔는데 실제로는 밤하늘이 소금빵에 대해 한 말이다.

    따옴표로 감싸지 않는 이유는 본문 인용 금지(규칙 4)와 충돌하기 때문이다.
    """
    log = "\n".join(
        f"[{u['t']}] {PHASE_KO.get(u['phase'], u['phase'])}"
        f" · 말한사람 {u['speaker']} · 발언 {u['text']}"
        for u in rows
    )
    return f"{log}\n{result_line(rounds, round_no)}"

print(render_np(day_log(ALL, 1), ROUNDS, 1)[-240:])   # 끝부분만 눈으로 확인


걸 왜 미리 말해요
[21:03:28] 투표 · 말한사람 라면왕 · 발언 다 들었나요
[21:03:45] 심판 · 말한사람 라면왕 · 발언 동률이라 AI 심판으로 넘어갔네요
[21:03:55] 심판 · 말한사람 딸기우유 · 발언 이거 누구 나오는 거예요
[21:04:05] 심판 · 말한사람 곰돌이 · 발언 초코 비 씨네요
[21:04:08] 심판 · 말한사람 초코비 · 발언 저 방학과 있습니다
결과: 처형 없음 (방어권_발동)


In [34]:
# ===== 셀 3 / 5 : 프롬프트 =====

SYSTEM_NP = """당신은 마피아 게임 전문 타블로이드 신문의 사회부 기자다.

입력은 마피아 게임의 낮 회의 기록이다.
출력은 다음 날 아침 배포될 조간 신문의 기사이다.

────────────────────────
목표
────────────────────────

이 기사는 회의를 빠짐없이 기록하는 문서가 아니다.

독자가 가장 먼저 읽고 싶어 할 장면만 골라
한 편의 기사처럼 편집한다.

좋은 기사는

- 누가 누구와 맞붙었는지
- 어느 한마디가 분위기를 뒤집었는지
- 투표와 최후변론, AI 심판 과정에서 어떤 일이 벌어졌는지

가 한눈에 보인다.

사소한 주장이나 반복되는 공방은 과감히 생략한다.

────────────────────────
기사 형식
────────────────────────

첫 줄은 ■ 로 시작하는 헤드라인.

회의에서 가장 큰 충돌이 있었다면
두 번째 헤드라인을 하나 더 추가할 수 있다.

헤드라인은

- 강한 발언
- 의외의 결과
- 가장 큰 충돌

중 하나를 기사체로 압축한다.

빈 줄 하나.

본문은 두 문단.

첫 문단은
회의에서 가장 극적이었던 장면 2~3개만 연결한다.

시간순으로 모두 요약하지 않는다.
가장 강한 장면부터 시작해도 된다.

둘째 문단은

- 투표
- 최후변론
- AI 심판

등 실제 일어난 결과만 간결하게 쓴다.

────────────────────────
문체
────────────────────────

신문 기사처럼 짧고 단단하게 쓴다.

짧은 문장과 긴 문장을 섞어 리듬을 만든다.

"~했다."만 반복하지 않는다.

본문에는 발언 인용을 1~3개 사용한다.

인용은

- 분위기를 바꾼 말
- 여러 사람의 시선을 모은 말
- 공격과 반격이 시작된 말

중에서 고른다.

인용 직후에는 반드시

- 다른 사람의 반응
또는
- 회의장 분위기 변화

를 이어서 쓴다.

기사는 회의를 요약하지 않는다.

독자가 "무슨 일이 있었길래?" 하고
계속 읽게 만드는 순서로 배치한다.

이름은 따옴표 없이 쓰고
항상 "님"을 붙인다.

────────────────────────
지킬 것
────────────────────────

누가 마피아인지 판단하지 않는다.

누가 누구를 의심했는지만 쓴다.

AI 심판 결과나 방어권 발동을
인물 평가와 연결하지 않는다.

주어진 발언 밖의 사실은 쓰지 않는다.

성별을 추측하지 않는다.

────────────────────────
좋은 기사의 기준
────────────────────────

다음 중 하나라도 만족하지 못하면 좋은 기사가 아니다.

- 회의의 핵심 갈등이 바로 보인다.
- 누가 누구를 몰아세웠는지 바로 보인다.
- 가장 인상적인 발언이 살아 있다.
- 회의 요약문이 아니라 신문 기사처럼 읽힌다.


────────────────────────
예시 1
────────────────────────

【예시】

■ "시간 낭비"라던 딸기우유 님, 하루 만에 최다 지목자로

"느낌으로 가야죠." — 딸기우유 님의 한마디에 회의장이 얼어붙었다.
곰돌이 님이 "그건 마피아가 제일 좋아하는 방식"이라 받아쳤고,
초코비 님도 곧바로 "제일 수상하다"며 손가락을 들었다.

밤하늘 님은 라면왕 님의 매끄러운 진행을 의심했고,
말 없던 소금빵 님도 그 시선을 피하지 못했다.

투표함이 열렸으나 셋이 나란히 섰다.
AI 심판이 부른 이름은 초코비 님.
그러나 폭탄은 터지지 않았다.

────────────────────────
예시 2
────────────────────────

【예시】

■ 마지막 한마디에 뒤집힌 시선… 조용하던 유자차 님, 회의 중심으로

회의 내내 말을 아끼던 유자차 님은
최후변론에서 "왜 저만 설명해야 하죠?"라고 짧게 되물었다.
순간 회의장의 공기가 달라졌고,
커피콩 님은 "오히려 그 반응이 더 이상하다"며 의심을 거두지 않았다.

침묵하던 참가자들까지 하나둘 의견을 내기 시작하면서
흩어졌던 시선은 다시 한 사람에게 모였다.

마지막 투표는 예상보다 한쪽으로 기울었다.
AI 심판은 가장 많은 표를 받은 참가자를 호명했고,
방어권 사용 여부를 두고 회의장은 다시 술렁였다.

────────────────────────
예시 3
────────────────────────

【예시】

■ 끝내 갈리지 않은 표심… 세 사람에게 쏠린 의심

"확신은 없어요."라는 말이 반복될수록
참가자들의 선택도 갈렸다.
초코라떼 님의 신중한 태도에
감귤 님은 "계속 안전한 말만 한다"고 맞받아쳤다.

한 사람에게 모일 것 같던 의심은
예상과 달리 여러 갈래로 흩어졌다.
회의 막판까지도 누구 하나 우위를 점하지 못했다.

투표 결과 세 사람이 비슷한 표를 받았다.
AI 심판이 최종 이름을 부르기 전까지
회의장은 누구도 결과를 쉽게 예상하지 못했다.

위 예시의 내용은 형식을 보여주기 위한 예시일 뿐,
실제 입력과 무관하다.
"""

USER_NP = """다음은 마피아 게임의 낮 회의 기록이다.

{log}

위 회의를 조간 타블로이드 신문 기사로 작성하시오.
"""

print("SYSTEM_NP", len(SYSTEM_NP), "· USER_NP", len(USER_NP))

SYSTEM_NP 2396 · USER_NP 60


In [35]:
# ===== 셀 4 / 5 : 호출 + 밤 로그 유출 검사 (TC-12) =====
# client·time 은 공통 0-1 에서 온다.
# 검사를 호출 안에 넣은 이유: TC-12 는 최우선 회귀 테스트라 빼먹을 수 없게 둔다.
#
# 밤 로그 경로가 노트북에서 등장하는 유일한 곳이다. 유출 검사 전용이며
# 프롬프트 입력에는 절대 닿지 않는다 (CLAUDE.md 의 자동화 예외 한 건).

NIGHT_LOG_NP = Path("log_analysis/scenario/scenario1/발화로그-밤마피아.txt")


def night_only_words(min_len=4):
    """밤 로그에만 나오는 어절. 낮에도 쓰는 어절은 오탐이라 뺀다.

    교집합을 빼지 않으면 "생각해요" 같은 것이 매 회차 걸려서 검사가 무시당한다.
    검사는 조용해야 쓸모가 있다.
    """
    night = set()
    for line in NIGHT_LOG_NP.read_text(encoding="utf-8").splitlines():
        if line.startswith("#") or ": " not in line:
            continue
        night |= set(line.split(": ", 1)[1].split())
    day = {w for u in ALL for w in u["text"].split()}
    return {w for w in night if len(w) >= min_len and w not in day}


NIGHT_WORDS = night_only_words()


def night_leak(text):
    """밤 발화 어절이 text 에 섞였는지 본다 (절대 규칙 1, 명세 21-2).

    눈으로는 놓치고, 한 줄만 새면 마피아 신원이 그대로 노출된다.
    불리언이 아니라 걸린 어절을 돌려주는 이유는 어디서 샜는지 보이게 하려는 것이다.
    """
    return sorted(w for w in NIGHT_WORDS if w in text)


def write_newspaper(round_no, utterances, rounds, model="gpt-4o-mini",
                    temperature=0.8, max_tokens=300, show_prompt=False):
    """R{round_no} 낮 → 제{round_no}호 본문.

    반환 (body, info)
      body ― LLM 출력. 머리글·사망자 줄이 아직 붙지 않은 상태다.
             밤에 조작권 소지자에게 보여주는 것이 이것 그대로다
      info ― sec·토큰·글자수·유출 검사

    max_tokens 가 글자수의 유일한 실효 상한이다. 실측 1토큰 ≈ 1.5자이므로
    300 이면 450자쯤에서 막힌다. 프롬프트의 글자수 지시는 지켜지지 않았다.

    sec 는 재시도 예산을 정한다. 생성 창이 R1 4초 / R2 9초 + 노출 지연 5초다.
    """
    rows = day_log(utterances, round_no)

    # 컷오프가 샌 경우를 코드로 먼저 막는다. 문자열 검사보다 이쪽이 정확하다.
    assert all(u["phase"] != "NIGHT" for u in rows), "밤 발화가 낮 로그에 섞였다"

    user = USER_NP.format(log=render_np(rows, rounds, round_no))
    leak_in = night_leak(user)
    if show_prompt:
        print("=== SYSTEM ===\n" + SYSTEM_NP)
        print("\n=== USER ===\n" + user + "\n")

    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,   # 창작이므로 0 이 아니다 → LLM호출설정 §4
        max_tokens=max_tokens,     # gpt-4o-mini 는 max_tokens.
                                   # gpt-5.x 로 바꾸면 max_completion_tokens 로
        messages=[{"role": "system", "content": SYSTEM_NP},
                  {"role": "user",   "content": user}],
    )
    body = resp.choices[0].message.content.strip()

    info = {"sec": time.perf_counter() - t0,
            "in":  resp.usage.prompt_tokens,
            "out": resp.usage.completion_tokens,
            "len": len(body),
            "leak_in":  leak_in,
            "leak_out": night_leak(body)}
    return body, info


print("밤 전용 어절", len(NIGHT_WORDS), "개")


밤 전용 어절 15 개


In [36]:
# ===== 셀 5 / 5 : 실행 =====
# 검사는 규칙으로 안 잡히는 것들이다. 규칙을 늘리기보다 코드로 세는 편이 낫다.

def masthead(round_no):
    """호수 = 기반이 된 낮의 라운드 번호. R1 낮 → R2 아침에 제1호.

    첫째 날 아침은 게임 시작 직후라 신문이 없다.
    호수를 서버가 붙이는 이유는 단순하다 ― 모델이 틀릴 일이 없어진다.
    """
    return f"【601 조간】 제{round_no}호"


def casualty_lines(rounds, round_no):
    """아침에 서버가 맨 앞에 붙이는 줄. LLM 은 절대 쓰지 않는다.

    밤에 마피아에게 보여줄 때는 만들 수 없다 ― 밤 살해가 아직 일어나지 않았다.
    역할 공개를 할 수 있는 주체가 서버뿐이므로 처형 줄이 필수다.
    맨 앞에 두는 이유는 17-6 ― 낮 시작에 10초만 표시한다.
    """
    r = rounds[round_no]
    lines = []
    if r["night_kill"]:
        lines.append(f"■ {r['night_kill']} 님이 마피아에게 살해당했습니다.")
    if r["execution"]:
        lines.append(f"■ {r['execution']} 님 처형 — {r['role']}였다")
    if not lines:
        lines.append("■ 평화로운 밤이었습니다")      # 17-7. 문구는 아직 미결
    return "\n".join(lines)


def check_np(body):
    """500자 초과만 본다 (명세 17-2).

    역할 단정·톤·인용은 눈으로 본다 ― 표현이 계속 바뀌어서 목록으로 못 잡는다.
    문자열 검사로 자동화하는 것은 밤 로그 유출뿐이고(CLAUDE.md), 그건 실행부에 있다.
    """
    if len(body) > 500:
        return [f"⚠️ {len(body)}자 (500 초과)"]
    return []



ROUND = 1      # R1 낮 → 제1호(둘째날 아침). 2 로 바꾸면 제2호 = 처형·역할 공개 케이스
RUNS  = 1      # 프롬프트 다듬는 동안은 1회. 문안 확정 뒤 마지막에 한 번만 3회로 본다

for i in range(1, RUNS + 1):
    body, info = write_newspaper(ROUND, ALL, ROUNDS)

    flags = check_np(body)
    if info["leak_in"] or info["leak_out"]:
        flags.insert(0, f"🔴 밤 로그 유출 {info['leak_in']} {info['leak_out']}")

    print("=" * 70)
    print(f"[{i}회차] {info['sec']:.1f}초 · 입력 {info['in']} · 출력 {info['out']}토큰 "
          f"· {info['len']}자")
    for f in flags:
        print("   ", f)
    print("=" * 70)
    print(f"{masthead(ROUND)}\n\n{body}\n")

# 아침 조립분은 사망자 줄만 확인한다. 본문은 위와 같으므로 다시 찍지 않는다.
print("[아침에 서버가 맨 앞에 붙일 줄]")
print(casualty_lines(ROUNDS, ROUND))


[1회차] 3.7초 · 입력 2427 · 출력 233토큰 · 350자
【601 조간】 제1호

■ 시민 안전을 외친 딸기우유 님, 첫 투표에서 동률로 AI 심판으로

"저는 시민이에요." — 딸기우유 님의 발언에 회의장은 잠시 굳어졌다. 그러나 곰돌이 님이 "딸기우유 씨가 아까부터 논의를 계속 막고 있어요"라고 반격하며 분위기는 다시 요동쳤다. 초코비 님도 "이번 투표는 딸기우유 씨한테 너를 생각입니다"라고 주장하며 의심의 화살을 날렸다. 이에 딸기우유 님은 "제가 뭘 막았다는 거죠?"라며 강하게 반발했다.

투표는 결국 동률로 마무리됐다. AI 심판이 개입하겠다고 선언하자, 참가자들 사이에 긴장감이 돌았다. 그러나 초코비 님은 방어권을 발동하며 처형은 이루어지지 않았다. 회의는 다시 복잡한 국면으로 접어들었다.

[아침에 서버가 맨 앞에 붙일 줄]
■ 소금빵 님이 마피아에게 살해당했습니다.


======================================================================
[1회차] 3.7초 · 입력 2427 · 출력 233토큰 · 350자
======================================================================
【601 조간】 제1호

■ 시민 안전을 외친 딸기우유 님, 첫 투표에서 동률로 AI 심판으로

"저는 시민이에요." — 딸기우유 님의 발언에 회의장은 잠시 굳어졌다. 그러나 곰돌이 님이 "딸기우유 씨가 아까부터 논의를 계속 막고 있어요"라고 반격하며 분위기는 다시 요동쳤다. 초코비 님도 "이번 투표는 딸기우유 씨한테 너를 생각입니다"라고 주장하며 의심의 화살을 날렸다. 이에 딸기우유 님은 "제가 뭘 막았다는 거죠?"라며 강하게 반발했다.

투표는 결국 동률로 마무리됐다. AI 심판이 개입하겠다고 선언하자, 참가자들 사이에 긴장감이 돌았다. 그러나 초코비 님은 방어권을 발동하며 처형은 이루어지지 않았다. 회의는 다시 복잡한 국면으로 접어들었다.

[아침에 서버가 맨 앞에 붙일 줄]
■ 소금빵 님이 마피아에게 살해당했습니다.

# 4. AI 심판

### 서버가 조회하는 것

1. 현재 생존자 목록 — 게임 상태. 사망자는 처형 후보가 될 수 없다
2. 낮 공개 발화 — Redis `utt:{roomId}:public`. 시각 · 라운드 · phase · 화자 · 본문
3. AI 심판 발동 시각 — 시나리오1 은 `21:03:39` (`PHASE R1/AI_JUDGMENT`)

### 서버가 LLM 에 넘기는 값

1. 현재 생존자 목록 (처형 후보) ← 조회 1
2. 생존자별 발화 통계 ← 조회 2 에서 **계산**
   발화 횟수 · 총 글자 수 · 전체 발화 비중 · 평균 발언 길이 · 라운드별 발화 횟수 · 다른 참가자 이름 언급 횟수
3. 심판 발동 시각 이전의 낮 공개 대화 ← 조회 2 에 조회 3 으로 컷오프

- 조회 3 은 프롬프트에 넣지 않는다. 2·3 의 컷오프 기준으로만 쓴다.
- 발화 통계는 별도 조회가 없다 — 발화 로그 하나에서 6종이 다 나온다.
- 심판 발동 시각은 프롬프트에 넣지 않는다. 3번의 컷오프 기준으로만 쓴다.

### 핵심 설계 원리

1. **역할을 넣지 않는다.** 절대 규칙 2. 칭호와 정반대다. 정답률이 비정상적으로 높으면 유출을 의심한다.
2. **컷오프를 코드로 한다.** `t < 심판 발동 시각`. 심판 구간 발화 4건이 자동으로 빠진다 — 그것들은 심판 결과에 대한 반응이라 입력이 되면 정답이 새어 나간다.
3. **통계는 서버가 세고 LLM 은 판결문만 쓴다.** 발화 로그 하나에서 통계 6종이 다 나와 별도 조회가 없다.
4. **통계는 생존자만, 낮 대화는 사망자 발화까지 넣는다.** 사망자는 판단 근거에 등장할 수 있지만 처형 대상은 될 수 없다.
5. **후보 밖의 이름은 코드가 버린다.** 프롬프트 품질과 무관하게 보장되는 환각 방어다.
6. **처형 대상은 마지막 한 문장으로 분리한다.** 판결 이유를 먼저 스트리밍하고, 대상은 이유가 끝난 뒤 별도 이벤트로 내보낸다. 그래야 대상이 반드시 마지막에 열린다.
7. **출력은 평문 판결문이다.** JSON 계약이 없다 — 칭호와 다른 점이다.
8. **진실 판정이 아니라 인상 평가다.** 논리적 모순과 거짓말 검증은 진술대조권 몫이고, 심판은 발화량·리듬·표현 반복만 본다.

### 알려진 한계

- 시나리오1 에서 AI 심판이 발동한 라운드는 R1 뿐이다. R2·R3 은 최다득표가 나와 최후변론으로 갔다.
- 그래서 라운드가 하나뿐이고, 평가 기준 2(발화량 변화)를 판단할 재료가 없다. 모델이 그것을 근거로 쓰면 없는 사실을 만든 것이다.


In [60]:
# ===== 셀 1 / 4 : 입력 준비 =====
# 서버가 조회하는 것 3개
#   ① 현재 생존자 목록  ② 낮 공개 발화  ③ AI 심판 발동 시각(컷오프)
# 공통 0-1 ~ 0-3 을 먼저 실행한다.
#
# 역할은 넣지 않는다 — 절대 규칙 2. 칭호와 정반대다.
# 심판은 발화만 보고 판단해야 하고, 정답률이 비정상적으로 높으면 유출을 의심한다.

ROUND_AJ = 1    # 시나리오1 에서 AI 심판이 발동한 라운드는 R1 뿐이다.
                # R2·R3 은 최다득표가 나와 최후변론으로 갔다


def load_state_AJ(round_no=ROUND_AJ):
    """AI 심판 발동 시각과 그 시점의 생존자를 뽑는다. 반환 (judged_at, alive)

    이벤트가 시간순이므로 심판 줄을 만나기 전에 나온 DEATH 만 센다.
    R1 밤 사망(21:04:42)은 심판(21:03:39) 이후라 자동으로 빠진다.
    ROLE 줄은 읽지 않는다 — 그 한 줄에 전원의 역할이 있다.
    """
    players, dead, judged_at = [], set(), None

    for t, rnd, phase, body in iter_events():
        g = re.match(r"GAME_START players=(\S+)", body)
        if g:
            players = g.group(1).split(",")
            continue

        if judged_at is None and rnd == round_no and phase == "AI_JUDGMENT":
            judged_at = t
            continue

        d = re.match(r"DEATH (\S+) ", body)
        if d and judged_at is None:
            dead.add(d.group(1))

    assert judged_at, f"R{round_no} 에 AI 심판이 없다"
    return judged_at, [p for p in players if p not in dead]


JUDGED_AT_AJ, ALIVE_AJ = load_state_AJ()

# 컷오프는 공통 before(). 심판 구간 발화 4건이 자동으로 빠진다 ―
# 그것들은 심판 결과에 대한 반응이라 입력이 되면 정답이 새어 나간다.
# phase 화이트리스트는 두지 않는다. 심판이 보는 것은 발화 리듬이라 시작 인사도 재료다.
PRE_AJ = before(ALL, JUDGED_AT_AJ)

print(f"R{ROUND_AJ} 심판 {JUDGED_AT_AJ} · 생존 {len(ALIVE_AJ)}명 {ALIVE_AJ}")
print(f"컷오프 이전 발화 {len(PRE_AJ)}건 · {sum(len(u['text']) for u in PRE_AJ)}자")


base_url : https://gms.ssafy.io/gmsapi/api.openai.com/v1/
R1 심판 21:03:39 · 생존 6명 ['라면왕', '밤하늘', '초코비', '소금빵', '곰돌이', '딸기우유']
컷오프 이전 발화 25건 · 738자


In [61]:
# ===== 셀 2 / 4 : 통계 계산 + 렌더 =====
# 발화 로그 하나에서 통계 6종이 다 나온다. 별도 조회가 없다.
# 세는 것은 서버가 하고 LLM 은 판결문만 쓴다.
#
# 정한 것 3개
#   비중 분모   ― 컷오프 이전 **전원** 발화 글자수 (사망자 포함)
#   DAY_START  ― 포함
#   이름 언급   ― 참가자 전원 이름, 자기 이름은 제외

# PHASE 라벨은 공통 0-2 의 PHASE_KO 를 쓴다.

def build_stats_AJ(utterances, alive, players):
    """{닉네임: 통계}. 생존자만 만든다 — 비교 대상이 생존자끼리다."""
    total_ch = sum(len(u["text"]) for u in utterances) or 1
    rounds = sorted({u["round"] for u in utterances})
    stats = {}

    for nick in alive:
        mine = [u for u in utterances if u["speaker"] == nick]
        ch = sum(len(u["text"]) for u in mine)
        stats[nick] = {
            "cnt":    len(mine),
            "chars":  ch,
            "share":  ch / total_ch * 100,
            "avg":    ch / len(mine) if mine else 0.0,
            "by_round": {r: sum(1 for u in mine if u["round"] == r) for r in rounds},
            "mentions": sum(u["text"].count(o) for u in mine
                            for o in players if o != nick),
        }
    return stats

def render_stats_AJ(stats):
    """받은 문안의 통계 예시 형식 그대로 맞춘다."""
    blocks = []
    for nick, s in stats.items():
        rounds = " / ".join(f"R{r} {c}회" for r, c in s["by_round"].items())
        blocks.append("\n".join([
            nick,
            f"- 발화 {s['cnt']}회, 총 {s['chars']}자",
            f"- 전체 공개 발화의 {s['share']:.1f}%",
            f"- 평균 발언 길이 {s['avg']:.1f}자",
            f"- {rounds}",
            f"- 다른 참가자 이름 언급 {s['mentions']}회",
        ]))
    return "\n\n".join(blocks)

def render_utterances_AJ(utterances):
    """[R1 토론 21:00:04] 라면왕: 자 첫 라운드니까 정보가 아예 없어요

    형식은 USER 프롬프트가 선언한 대로 `화자: 발화 내용` 이다.
    주의 ― 조간신문에서는 이 형식이 주체를 뒤집었다 (한 줄에 이름이 둘). 
    판결 이유에 "이 참가자"만 쓰게 되어 있어 영향은 작지만, 
    엉뚱한 사람의 특징이 근거로 적히는지 눈으로 확인한다.
    """
    return "\n".join(
        f"[R{u['round']} {PHASE_KO.get(u['phase'], u['phase'])} {u['t']}]"
        f" {u['speaker']}: {u['text']}"
        for u in utterances
    )

STATS_AJ = build_stats_AJ(PRE_AJ, ALIVE_AJ, ALIVE_AJ)

print(render_stats_AJ(STATS_AJ))


라면왕
- 발화 6회, 총 128자
- 전체 공개 발화의 17.3%
- 평균 발언 길이 21.3자
- R1 6회
- 다른 참가자 이름 언급 1회

밤하늘
- 발화 3회, 총 80자
- 전체 공개 발화의 10.8%
- 평균 발언 길이 26.7자
- R1 3회
- 다른 참가자 이름 언급 1회

초코비
- 발화 2회, 총 100자
- 전체 공개 발화의 13.6%
- 평균 발언 길이 50.0자
- R1 2회
- 다른 참가자 이름 언급 1회

소금빵
- 발화 2회, 총 28자
- 전체 공개 발화의 3.8%
- 평균 발언 길이 14.0자
- R1 2회
- 다른 참가자 이름 언급 0회

곰돌이
- 발화 5회, 총 200자
- 전체 공개 발화의 27.1%
- 평균 발언 길이 40.0자
- R1 5회
- 다른 참가자 이름 언급 4회

딸기우유
- 발화 7회, 총 202자
- 전체 공개 발화의 27.4%
- 평균 발언 길이 28.9자
- R1 7회
- 다른 참가자 이름 언급 1회


In [62]:
# ===== 셀 3 / 4 : 프롬프트 =====
# SYSTEM_AJ · USER_AJ 는 받은 문안 그대로다. 문구를 고치지 않는다.
# 출력이 평문 판결문이라 JSON 계약을 덧붙일 필요가 없다 — 칭호와 다른 점이다.
#
# 주의: SYSTEM_AJ 에 .format() 을 쓰지 않는다. 마지막 문장 형식의 중괄호가 깨진다.
#      값을 끼우는 것은 USER_AJ 쪽뿐이다.

SYSTEM_AJ = """당신은 마피아 게임의 AI 심판이다.

당신의 임무는 실제 마피아를 정확히 찾아내는 것이 아니다.
게임 중 공개된 낮 대화를 관찰하여, 회의 참여 방식과 말투가 가장 눈에 띄거나 수상한 생존자 한 명을 처형 대상으로 선정하는 것이다.

이 판단은 진실 판정이 아니라 회의 분위기에 대한 불완전한 인상 평가다.
적극적으로 추리한 시민, 긴장한 시민, 말이 적었던 시민도 처형 대상으로 선정될 수 있다.

────────────────────────
평가 기준
────────────────────────

다음 요소를 종합적으로 평가한다.

1. 발화 존재감
- 다른 참가자에 비해 지나치게 많이 또는 적게 말했는가
- 한 사람의 발언 비중이 유난히 높거나 낮은가
- 평균 발언 길이가 다른 참가자와 크게 다른가

2. 발화량 변화
- 조용하다가 특정 시점부터 갑자기 발언이 늘었는가
- 활발하게 말하다가 투표 직전에 갑자기 조용해졌는가
- 라운드가 바뀌면서 발언 횟수나 길이가 크게 변했는가

3. 표현 반복과 자기 정정
- 같은 단어나 표현을 반복했는가
- 비슷한 말을 여러 번 되풀이했는가
- 말을 시작했다가 곧바로 다른 표현으로 고쳤는가
- 짧은 해명이나 반박을 연속으로 내놓았는가

4. 강한 표현과 방어적 반응
- 짧고 단호한 반박이 반복됐는가
- 자신을 향한 의심이나 질문 이후 강한 표현이 늘었는가
- 다른 참가자에게 책임을 돌리는 표현이 두드러졌는가
- 특정 참가자의 이름을 지나치게 자주 언급했는가

5. 대화 리듬의 변화
- 평소의 발언 형태와 다른 갑작스러운 변화가 있었는가
- 투표가 가까워질수록 발언이 급격히 짧아지거나 많아졌는가
- 특정 참가자와의 공방에 지나치게 집중했는가

────────────────────────
판단 원칙
────────────────────────

- 실제 역할은 추측하지 않는다.
- 누가 마피아인지 단정하지 않는다.
- 발언 내용의 사실 여부를 판단하지 않는다.
- 서로 다른 발언의 논리적 모순을 찾지 않는다.
- 알리바이나 거짓말을 검증하지 않는다.
- 투표가 정확했는지 평가하지 않는다.
- 진술대조 분석처럼 과거 발언과 현재 발언을 논리적으로 비교하지 않는다.
- "화를 냈다", "거짓말했다", "말을 더듬었다"라고 단정하지 않는다.
- 대신 "강한 표현이 늘었다", "같은 표현을 반복했다", "짧은 반박이 이어졌다"처럼 관찰 가능한 현상만 말한다.
- 서버가 제공한 수치와 공개된 낮 대화 밖의 사실을 만들지 않는다.
- 밤에 있었던 대화나 행동을 추측하지 않는다.
- 사망자는 판단 근거에 등장할 수 있지만 처형 대상으로 선택할 수 없다.
- 처형 대상은 반드시 후보 목록에 있는 현재 생존자여야 한다.

가장 논리적으로 의심스러운 사람을 찾지 말고,
회의 참여 방식이 다른 생존자와 비교했을 때 가장 불안정하거나 눈에 띈 사람을 선택한다.

발화량이 많다는 이유 하나만으로 선정하지 않는다.
발화량, 발화량 변화, 표현 반복, 강한 반응, 이름 언급 집중도 중 적어도 두 가지 이상의 특징을 함께 고려한다.

────────────────────────
판결문 작성
────────────────────────

판결문은 3단계로 작성한다.

1. 처음에는 관찰된 발화 특징을 차분하게 설명한다.
2. 이어서 그것이 정상적인 긴장이나 적극적인 추리였을 가능성도 인정한다.
3. 마지막 문장에서만 처형 대상을 공개한다.

선정된 대상의 닉네임을 판결 이유에서 미리 밝히지 않는다.
"이 참가자", "해당 참가자"처럼 표현한다.

판결 이유는 3~5문장으로 작성한다.
문장은 짧고 무게감 있게 쓴다.
과도하게 확신하지 않되, 최종 판결은 단호하게 발표한다.

마지막 문장은 반드시 아래 형식과 정확히 같아야 한다.

처형 대상은 {닉네임}입니다.

마지막 문장 뒤에는 어떤 설명도 덧붙이지 않는다.
제목, 목록, JSON, 마크다운은 출력하지 않는다.
판결문 본문만 출력한다."""

USER_AJ = """다음은 AI 심판 시작 시점 이전까지 수집된 게임 정보다.

────────────────────────
현재 생존자 — 처형 후보
────────────────────────

{alive_players}

────────────────────────
생존자별 발화 통계
────────────────────────

{player_statistics}

통계에는 다음 정보가 포함될 수 있다.

- 전체 발화 횟수
- 전체 발화 글자 수
- 전체 대화에서 차지하는 발화 비중
- 평균 발언 길이
- 라운드별 발화 횟수
- 다른 참가자 닉네임 언급 횟수

────────────────────────
공개된 낮 대화
────────────────────────

{public_day_utterances}

각 발화는 다음 형식이다.

[라운드 페이즈 시각] 화자: 발화 내용

위 정보만 이용하여 현재 생존자 중 한 명을 처형 대상으로 선정하라.

발언의 진실 여부나 논리적 모순을 분석하지 말고,
다른 참가자와 비교했을 때 나타나는 발화량, 발화량 변화, 표현 반복, 강한 반응, 이름 언급 집중도와 대화 리듬을 평가하라.

판결 이유를 먼저 서술하고 마지막 문장에서만 처형 대상을 공개하라."""

print("SYSTEM_AJ", len(SYSTEM_AJ), "· USER_AJ", len(USER_AJ))


SYSTEM_AJ 1930 · USER_AJ 614


In [65]:
# ===== 셀 4 / 4 : 호출 + 검증 + 실행 =====

VERDICT_RE_AJ = re.compile(r"처형 대상은\s*(.+?)\s*입니다\.?\s*$")


def split_verdict_AJ(text):
    """판결 이유와 처형 대상을 분리한다. 반환 (reason, target)

    실서비스의 스트리밍 계약이 여기서 정해진다 ―
    "처형 대상은" 을 만나면 그 줄을 붙잡아 두고, 이유 출력이 끝난 뒤
    별도 judgment.verdict 이벤트로 내보낸다. 그래야 대상이 반드시 마지막에 열린다.
    """
    body = text.strip()
    m = VERDICT_RE_AJ.search(body)
    if not m:
        return body, None
    return body[:m.start()].strip(), m.group(1)


def check_AJ(reason, target, alive):
    """계약 위반만 센다. 판결문의 톤과 설득력은 눈으로 본다."""
    problems = []
    if target is None:
        return ["마지막 문장이 '처형 대상은 OOO입니다.' 형식이 아니다"]

    # 없는 이름은 버린다 — 진술대조의 "없는 번호 버림" 과 같은 계열의 환각 방어.
    # 프롬프트를 어떻게 고쳐도 후보 밖의 이름은 여기서 걸린다.
    if target not in alive:
        problems.append(f"후보에 없는 대상 '{target}' (생존 {alive})")

    # 규칙 ― 선정된 닉네임을 판결 이유에서 미리 밝히지 않는다
    if target in reason:
        problems.append(f"판결 이유에 '{target}' 이 미리 등장했다")

    n = len([s for s in re.split(r"[.!?]\s*", reason) if s.strip()])
    if not 3 <= n <= 5:
        problems.append(f"판결 이유 {n}문장 (3~5 벗어남)")

    if any(ch in reason for ch in ("#", "**", "{", "- ")):
        problems.append("마크다운·목록·JSON 흔적")

    return problems


def judge_AJ(alive, stats, utterances, model="gpt-5.4-mini",
             temperature=0.7, max_completion_tokens=2048, show_prompt=False):
    """반환 (raw, reason, target, problems, info)

    max_completion_tokens 를 넉넉히 두는 이유는 진술대조와 같다 ―
    gpt-5.x 는 추론 토큰이 이 상한을 같이 먹는다. 상한은 잘림 방지용이라
    과하게 둬도 손해가 없다.

    temperature 는 0 이 아니다 ― 판결문은 창작이다. 그래서 회차마다 달라진다.
    """
    user = USER_AJ.format(
        alive_players="\n".join(f"- {n}" for n in alive),
        player_statistics=render_stats_AJ(stats),
        public_day_utterances=render_utterances_AJ(utterances),
    )
    if show_prompt:
        print("=== SYSTEM ===\n" + SYSTEM_AJ)
        print("\n=== USER ===\n" + user + "\n")

    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,   # gpt-5.x 는 max_tokens 를 주면 400
        messages=[{"role": "system", "content": SYSTEM_AJ},
                  {"role": "user",   "content": user}],
    )
    info = {"sec": time.perf_counter() - t0,
            "in":  resp.usage.prompt_tokens,
            "out": resp.usage.completion_tokens}

    raw = resp.choices[0].message.content.strip()
    reason, target = split_verdict_AJ(raw)
    return raw, reason, target, check_AJ(reason, target, alive), info


RUNS_AJ = 1     # 1회로 판단하지 않는다. 문안 확정 뒤 3회로 본다.
                # 실제 서버 판정은 초코비였다 (AI_JUDGMENT_TARGET, 방어권으로 무효)

for i in range(1, RUNS_AJ + 1):
    raw, reason, target, problems, info = judge_AJ(ALIVE_AJ, STATS_AJ, PRE_AJ)

    print("=" * 70)
    print(f"[{i}회차] {info['sec']:.1f}초 · 입력 {info['in']} · 출력 {info['out']}토큰"
          f" · 대상 {target}")
    for p in problems:
        print("   ⚠️", p)
    print("=" * 70)
    print("[판결 이유 — 먼저 스트리밍한다]")
    print(reason)
    print()
    print("[judgment.verdict — 이유가 끝난 뒤 따로 내보낸다]")
    print(f"처형 대상은 {target}입니다.")
    print()


[1회차] 1.8초 · 입력 2589 · 출력 116토큰 · 대상 곰돌이
[판결 이유 — 먼저 스트리밍한다]
이 참가자는 발화량이 가장 많고, 짧은 반응과 진행형 발언이 여러 차례 이어졌습니다. 특히 특정 참가자 이름을 반복해서 언급하며 공방의 중심에 오래 머문 점이 눈에 띕니다. 다만 라운드 초반에 분위기를 주도하려는 적극적인 태도였을 가능성도 충분히 있습니다. 그럼에도 회의 리듬을 가장 크게 흔든 쪽으로 보입니다.

[judgment.verdict — 이유가 끝난 뒤 따로 내보낸다]
처형 대상은 곰돌이입니다.



======================================================================
결과 : [1회차] 1.8초 · 입력 2589 · 출력 116토큰 · 대상 곰돌이
======================================================================
[판결 이유 — 먼저 스트리밍한다]
이 참가자는 발화량이 가장 많고, 짧은 반응과 진행형 발언이 여러 차례 이어졌습니다. 특히 특정 참가자 이름을 반복해서 언급하며 공방의 중심에 오래 머문 점이 눈에 띕니다. 다만 라운드 초반에 분위기를 주도하려는 적극적인 태도였을 가능성도 충분히 있습니다. 그럼에도 회의 리듬을 가장 크게 흔든 쪽으로 보입니다.

[judgment.verdict — 이유가 끝난 뒤 따로 내보낸다]
처형 대상은 곰돌이입니다.

# 5. 칭호부여

### 서버가 조회하는 것

1. 참가자 닉네임 목록
2. 플레이어별 역할 (시민 / 마피아)
3. 사망 기록 — 누가 · 몇 라운드 · 원인 · 그때의 phase
4. 최종 승리 진영과 생존자
5. 낮 공개 발화 — Redis `utt:{roomId}:public`

### 서버가 LLM 에 넘기는 값

1. 최종 승리 진영 ← 조회 4
2. 플레이어별 역할 (시민 / 마피아) ← 조회 2
3. 플레이어별 사망 라운드와 원인 (마피아 살해 / AI 심판 처형 / 최후변론 투표 처형), 생존자는 생존 표시 ← 조회 3·4
4. 낮 공개 대화 전체 (시각 · 라운드 · phase · 화자 · 본문) ← 조회 5

승패는 넘기는 값에 없다. 조회 4 + 조회 2 로 코드가 계산한다.
조회 3 의 phase 도 넘기지 않는다 — 사망 원인 3종을 가르는 데만 쓴다.

### 주요 설계 원리

1. **낮 대화로 알 수 있는 것은 넘기지 않는다.** 발화량·득표·투표 내역·이름 언급 횟수를 빼면 모델이 그 숫자를 틀릴 일도 없어진다.
2. **승패는 코드가 계산한다.** 승리 진영 + 역할의 결과라 새 정보가 아니다. 모델이 매번 역산하게 두지 않는다.
3. **역할을 넣는 유일한 기능이다.** 게임 종료 = 역할 공개 시점이므로 절대 규칙 2의 예외다.
4. **밤 대화는 넣지 않고 경로도 선언하지 않는다.** 밤 구간의 서버 이벤트(사망)만 쓴다. 금지 대상은 대화 원문이다.
5. **사망 원인은 `phase` 로 가른다.** AI 심판 처형과 최후변론 처형이 이벤트 로그에서 둘 다 `cause=EXECUTION` 이다.
6. **전원을 한 번의 호출로 받는다.** 칭호 중복 금지가 계약이라 서로를 봐야 한다. 게임당 1회 호출이다.
7. **출력 계약은 프롬프트에 직접 적는다.** 받은 문안에 형식 지시가 없어 `OUTPUT_TT` 로 분리해 뒤에 붙였다. 본문은 손대지 않는다.
8. **코드 검증은 계약 위반만 센다.** 6명 정확히 한 번 · 없는 닉네임 · 칭호 중복 · 빈 항목. 재미와 톤은 눈으로 본다.
9. **`temperature=0.9` 에 회차를 비교한다.** 창작이므로 0이 아니고, 그래서 한 번 돌려서 판단하지 않는다.


In [54]:
# ===== 셀 1 / 5 : 입력 준비 =====
# 서버가 넘기는 것은 4개뿐이다.
#   ① 최종 승리 진영  ② 플레이어별 역할  ③ 사망 라운드와 원인  ④ 낮 공개 대화 전체
# 발화량·득표·투표 내역·아이템처럼 낮 대화로 셀 수 있거나 추론할 수 있는 것은
# 넘기지 않는다. LLM 이 대본을 보고 판단한다.
# 공통 0-1 ~ 0-3 을 먼저 실행한다.
#
# ROLE 줄을 읽는 유일한 기능이다 — 게임 종료 = 역할 공개 시점 (절대 규칙 2 의 예외).
# 그래도 밤 대화는 넣지 않는다. 금지 대상은 밤의 대화 원문이다.


def parse_result_TT():
    """서버가 확정한 4가지만 뽑는다.

    반환 dict
      players   ― 참가자 닉네임 순서
      roles     ― {닉네임: "시민"|"마피아"}
      deaths    ― {닉네임: {round, cause, phase}}
      winner    ― "시민"|"마피아"
      survivors ― [닉네임]

    VOTE_CAST·VOTE_TALLY·ITEM_USE 는 읽지 않는다 — 이번 설계에서는 LLM 이
    낮 대화의 투표 발언과 아이템 사용 발언으로 판단한다.
    DEATH 의 phase 를 같이 담는 이유는 아래 death_label_TT 참조.
    줄 순회는 공통 iter_events() 가 한다.
    """
    res = {"players": [], "roles": {}, "deaths": {},
           "winner": None, "survivors": []}

    for _t, rnd, phase, body in iter_events():
        g = re.match(r"GAME_START players=(\S+)", body)
        if g:
            res["players"] = g.group(1).split(",")
            continue

        if body.startswith("ROLE "):
            for pair in body[5:].split():
                nick, role = pair.split("=")
                res["roles"][nick] = role
            continue

        d = re.match(r"DEATH (\S+) cause=(\S+)", body)
        if d:
            res["deaths"][d.group(1)] = {"round": rnd, "cause": d.group(2),
                                         "phase": phase}
            continue

        e = re.match(r"GAME_END winner=(\S+) survivors=(\S+)", body)
        if e:
            res["winner"] = e.group(1)
            res["survivors"] = e.group(2).split(",")
    return res


def death_label_TT(d):
    """사망 원인 3종. 처형은 phase 로 갈린다.

    AI 심판이 지목해 죽은 경우와 최후변론 투표로 죽은 경우가 이벤트 로그에서는
    둘 다 cause=EXECUTION 이다. 구분되는 것은 그때의 phase 뿐이다.
    시나리오1 에는 AI 심판 사망이 없다 — 초코비가 방어권으로 막았다.
    """
    if d["cause"] == "NIGHT_KILL":
        return "마피아에게 살해당함"
    if d["cause"] == "EXECUTION" and d["phase"] == "AI_JUDGMENT":
        return "AI 심판 지목으로 처형됨"
    if d["cause"] == "EXECUTION":
        return "최후변론 투표로 처형됨"
    return d["cause"]


DAY_TT = ALL              # 칭호는 낮 공개 대화 전체를 본다. 컷오프가 없다
RES_TT = parse_result_TT()

print(f"낮 공개 발화 {len(DAY_TT)}건 · {sum(len(u['text']) for u in DAY_TT)}자")
print("승리", RES_TT["winner"], "· 생존", RES_TT["survivors"])
for nick in RES_TT["players"]:
    d = RES_TT["deaths"].get(nick)
    state = "끝까지 생존" if not d else f"R{d['round']} {death_label_TT(d)}"
    print(f"  {nick:<5} {RES_TT['roles'][nick]:<3} {state}")


base_url : https://gms.ssafy.io/gmsapi/api.openai.com/v1/
낮 공개 발화 79건 · 1967자
승리 시민 · 생존 ['라면왕', '곰돌이']
  라면왕   시민  끝까지 생존
  밤하늘   시민  R2 마피아에게 살해당함
  초코비   마피아 R3 최후변론 투표로 처형됨
  소금빵   시민  R1 마피아에게 살해당함
  곰돌이   시민  끝까지 생존
  딸기우유  마피아 R2 최후변론 투표로 처형됨


In [55]:
# ===== 셀 2 / 5 : 렌더 =====
# USER 프롬프트가 요구하는 형식에 맞춘다.
#   플레이어 정보 ― 닉네임 · 역할 · 승패 · 생존 여부
#   낮 대화      ― [라운드 페이즈 시각] 화자: 발화 내용
#
# 승패는 서버 전달값에 없지만 "승리 진영 + 역할" 의 결과라 새 정보가 아니다.
# 그래서 코드가 만든다 — 모델이 매번 역산하게 두면 틀릴 여지만 생긴다.

# PHASE 라벨은 공통 0-2 의 PHASE_KO 를 쓴다.

def render_players_TT(res):
    """플레이어당 4줄. 서버가 확정한 것만 적는다.

    발화량·득표·투표 횟수는 넣지 않는다. 낮 대화에 다 있고,
    같은 사실을 두 번 주면 모델이 어느 쪽을 근거로 삼았는지 알 수 없다.
    """
    blocks = []
    for nick in res["players"]:
        role = res["roles"][nick]
        d = res["deaths"].get(nick)
        state = "끝까지 생존" if not d else f"R{d['round']}에 사망 — {death_label_TT(d)}"
        blocks.append("\n".join([
            f"■ {nick}",
            f"  역할: {role}",
            f"  결과: {'승리' if role == res['winner'] else '패배'}",
            f"  생존: {state}",
        ]))
    return "\n\n".join(blocks)


def render_utterances_TT(utterances):
    """[R1 토론 21:00:04] 라면왕: 자 첫 라운드니까 정보가 아예 없어요

    형식은 USER 프롬프트가 선언한 대로 `화자: 발화 내용` 이다.
    주의 ― 조간신문에서는 이 형식이 문제를 일으켜 `말한사람 X · 발언 Y` 로
    바꿨다. 한 줄에 이름이 둘(화자 + 언급 대상)이면 모델이 주체를 뒤집는다
    (2026-07-30 실측, 2회 연속). 프롬프트를 그대로 쓰기로 했으므로 이대로 두고,
    시상평에 주체가 뒤집히는지 눈으로 확인한다.
    """
    return "\n".join(
        f"[R{u['round']} {PHASE_KO.get(u['phase'], u['phase'])} {u['t']}]"
        f" {u['speaker']}: {u['text']}"
        for u in utterances
    )


# {votes} 자리 ― 서버 전달값에서 투표 내역이 빠졌으므로 집계를 넣지 않는다.
# 낮 대화의 DAY_VOTE 구간에 "저는 초코비 씨입니다" 같은 투표 발언이 남아 있어
# 모델이 그것으로 판단한다. 서버 집계를 다시 주기로 하면 이 상수만 바꾸면 된다.
VOTES_TT = "(서버 투표 집계는 제공하지 않는다. 낮 대화의 투표 구간 발언을 참고하라.)"

print(render_players_TT(RES_TT))
print()
print(render_utterances_TT(DAY_TT)[:300], "...")


■ 라면왕
  역할: 시민
  결과: 승리
  생존: 끝까지 생존

■ 밤하늘
  역할: 시민
  결과: 승리
  생존: R2에 사망 — 마피아에게 살해당함

■ 초코비
  역할: 마피아
  결과: 패배
  생존: R3에 사망 — 최후변론 투표로 처형됨

■ 소금빵
  역할: 시민
  결과: 승리
  생존: R1에 사망 — 마피아에게 살해당함

■ 곰돌이
  역할: 시민
  결과: 승리
  생존: 끝까지 생존

■ 딸기우유
  역할: 마피아
  결과: 패배
  생존: R2에 사망 — 최후변론 투표로 처형됨

[R1 낮시작 21:00:04] 라면왕: 이 시작했네요
[R1 토론 21:00:08] 라면왕: 자 첫 라운드니까 정보가 아예 없어요 일단 한 명씩 돌아가면서 얘기해보죠
[R1 토론 21:00:15] 딸기우유: 저는 그런 거 시간 남비라고 생각해요 그냥 느낌으로 가야죠
[R1 토론 21:00:23] 곰돌이: 느낌으로 가면 시민만 죽어요 그건 마피아가 제일 좋아하는 방식이고요
[R1 토론 21:00:31] 딸기우유: 아니 그러면 3년 동안 뭘 어떻게 하겠다는 거예요 근데 신이라고 할 거잖아요
[R1 토론 21:00:40] 밤하늘: 저는 ...


In [56]:
# ===== 셀 3 / 5 : 프롬프트 =====
# SYSTEM_BASE_TT · USER_TT 는 받은 문안 그대로다. 문구를 고치지 않는다.
#
# OUTPUT_TT 를 따로 만든 이유 ―
#   받은 SYSTEM·USER 에는 출력 형식 지시가 없다. 그대로 보내면 JSON 이 아니라
#   줄글이 온다. "출력 형식" 절에 있던 계약을 그 자리에 넣어 뒤에 붙인다.
#   분리해 두면 나중에 문안을 갱신할 때 어디가 원본인지 헷갈리지 않는다.
#
# 주의: SYSTEM_TT 에 .format() 을 쓰지 않는다. OUTPUT_TT 의 중괄호가 깨진다.
#      값을 끼우는 것은 USER_TT 쪽뿐이다.

SYSTEM_BASE_TT = """당신은 마피아 게임이 끝난 뒤 시상식을 진행하는 재치 있는 사회자다.

입력에는 서버가 확정한 게임 결과와 공개된 낮 대화가 제공된다.

모든 플레이어의 실제 플레이를 살펴보고, 각 플레이어에게 가장 잘 어울리는 칭호를 정확히 하나씩 부여하라.

────────────────────────
칭호 선정 기준
────────────────────────

다음 요소를 종합적으로 살펴본다.

1. 추리와 투표
- 마피아를 정확하게 의심하거나 투표했는가
- 시민을 반복해서 의심하거나 투표했는가
- 같은 대상을 꾸준히 의심했는가
- 라운드마다 의심 대상을 자주 바꿨는가
- 다른 사람보다 먼저 정확한 의심을 제기했는가

2. 회의 참여 방식
- 발화량이 다른 참가자보다 많거나 적었는가
- 다른 참가자의 이름을 자주 언급했는가
- 여러 사람을 공격하거나 의심했는가
- 다른 참가자를 적극적으로 옹호했는가
- 자신의 의견이 다른 참가자에게 영향을 주었는가

3. 역할과 경기 결과
- 시민으로서 억울하게 많은 의심을 받았는가
- 마피아로서 의심받지 않고 오래 살아남았는가
- 마피아가 같은 마피아에게 투표했는가
- 발화량이 적었지만 마피아로 승리했는가
- 실제 역할과 플레이 방식 사이에 재미있는 대비가 있는가

4. 아이템과 생존
- 아이템을 적극적으로 사용했는가
- 중요한 순간에 아이템 효과를 발동했는가
- 아이템을 끝까지 사용하지 않았는가
- 일찍 사망했는가
- 마지막까지 생존했는가

────────────────────────
작성 규칙
────────────────────────

- 모든 플레이어에게 칭호를 정확히 하나씩 부여한다.
- 입력에 있는 모든 플레이어를 빠짐없이 포함한다.
- 서로 다른 플레이어에게 같은 칭호를 부여하지 않는다.
- 서버가 제공한 역할, 승패, 투표, 생존, 아이템 기록을 변경하지 않는다.
- 공개된 낮 대화와 서버 기록에 근거가 있는 칭호만 부여한다.
- 밤 대화나 밤에 있었을 법한 발언을 추측하지 않는다.
- 입력에 없는 사건, 발언, 횟수를 만들지 않는다.
- 단순히 시민 또는 마피아라는 역할만 보고 칭호를 정하지 않는다.
- 플레이어의 구체적인 행동과 역할 사이의 재미있는 관계를 찾는다.
- 예시 칭호를 그대로 사용해도 되고, 실제 플레이에 더 잘 맞는 새로운 칭호를 만들어도 된다.
- 칭호는 짧고 기억하기 쉽게 작성한다.
- 모욕적이거나 불쾌한 표현은 피한다.
- 모든 참가자가 함께 웃을 수 있는 게임 시상식 분위기로 작성한다.

quote는 해당 플레이어에게 직접 건네는 한 문장짜리 시상평이다.

reason은 칭호 선정 근거다.
역할, 투표, 발화, 아이템, 생존 기록 중 실제로 확인되는 내용을 짧고 구체적으로 적는다.

대화 원문을 길게 복사하지 않는다."""


OUTPUT_TT = """

────────────────────────
출력 형식
────────────────────────

아래 JSON 형식으로만 답한다. 설명이나 코드블록 표시를 붙이지 않는다.

{"titles": [{"nickname": "라면왕", "title": "뚝심의 추리왕", "quote": "첫 의심을 끝까지 놓지 않은 집념이 결국 마을을 구했습니다.", "reason": "초코비를 꾸준히 의심하고 마지막 라운드에 실제로 투표해 시민 승리에 기여했다."}]}

titles 는 참가자 수와 같은 길이의 배열이다.
nickname 은 입력에 있는 닉네임을 글자 그대로 쓴다."""


SYSTEM_TT = SYSTEM_BASE_TT + OUTPUT_TT


USER_TT = """다음은 한 판의 마피아 게임이 종료된 뒤 서버가 제공한 게임 결과와 공개된 낮 대화다.

────────────────────────
게임 결과
────────────────────────

승리 진영:
{winner}

────────────────────────
플레이어 정보
────────────────────────

{players}

각 플레이어 정보에는 다음 내용이 포함될 수 있다.

- 닉네임
- 역할
- 승리 또는 패배
- 생존 여부
- 발화 횟수와 발화량
- 받은 표 수
- 마피아에게 투표한 횟수
- 시민에게 투표한 횟수
- 투표 대상 변경 횟수
- 아이템 사용 내역
- 다른 플레이어 이름 언급 횟수

────────────────────────
라운드별 투표 내역
────────────────────────

{votes}

────────────────────────
공개된 낮 대화 전체
────────────────────────

{public_day_utterances}

각 발화는 다음 형식이다.

[라운드 페이즈 시각] 화자: 발화 내용

────────────────────────
칭호 예시
────────────────────────

역대급 억울함
완벽한 추리
소름 돋는 거짓말
웅변가
침묵의 암살자
인간 확성기
의심 제조기
국민 변호사
여론 장악자
갈대왕
뚝심의 추리왕
무고한 희생양
자백 직전상
촉이 왔다상
완전범죄
최고의 팀킬상

위 기록을 바탕으로 모든 플레이어에게 가장 잘 어울리는 칭호를 하나씩 부여하라.

예시보다 더 잘 어울리는 칭호가 있다면 새로 만들어도 된다."""

print("SYSTEM_TT", len(SYSTEM_TT), "(본문", len(SYSTEM_BASE_TT),
      "+ 출력계약", len(OUTPUT_TT), ") · USER_TT", len(USER_TT))


SYSTEM_TT 1678 (본문 1356 + 출력계약 322 ) · USER_TT 799


In [57]:
# ===== 셀 4 / 5 : 호출 + JSON 파싱 + 검증 =====
# 검증은 계약 위반만 센다. 재미와 톤, 사실 충돌은 눈으로 본다.
# 어서션 러너를 만들지 않는다 (CLAUDE.md) — 기대답안은 참조 샘플이다.

# strip_fence 는 공통 0-4 것을 쓴다.

def check_titles_TT(parsed, players):
    """돌려주는 것이 불리언이 아니라 문제 목록인 이유는 진술대조와 같다 ―
    어디가 깨졌는지 보여야 프롬프트를 어디를 고칠지 알 수 있다.
    """
    if not isinstance(parsed, dict):
        return ["JSON 객체가 아니다"]
    titles = parsed.get("titles")
    if not isinstance(titles, list):
        return ["titles 가 배열이 아니다"]

    problems = []
    rows = [t for t in titles if isinstance(t, dict)]
    if len(rows) != len(titles):
        problems.append("titles 안에 객체가 아닌 항목이 있다")

    got = [t.get("nickname") for t in rows]
    missing = [p for p in players if got.count(p) == 0]
    twice   = [p for p in players if got.count(p) > 1]
    unknown = [n for n in got if n not in players]
    if missing:
        problems.append(f"빠진 참가자 {missing}")
    if twice:
        problems.append(f"두 번 나온 참가자 {twice}")
    if unknown:
        problems.append(f"없는 닉네임 {unknown}")

    names = [t.get("title") for t in rows]
    dup = sorted({n for n in names if names.count(n) > 1})
    if dup:
        problems.append(f"칭호 중복 {dup}")

    for t in rows:
        blank = [k for k in ("nickname", "title", "quote", "reason")
                 if not str(t.get(k) or "").strip()]
        if blank:
            problems.append(f"{t.get('nickname') or '?'} 의 빈 항목 {blank}")
    return problems


def award_titles_TT(res, utterances, model="gpt-4o-mini",
                    temperature=0.9, max_tokens=1500, show_prompt=False):
    """전원의 칭호를 한 번의 호출로 받는다. 게임당 1회다.

    반환 (raw, parsed, problems, info)

    전원을 한 번에 보내는 이유 ― 칭호 중복 금지가 계약이다.
    한 명씩 6번 호출하면 모델이 다른 사람에게 무엇을 줬는지 모른다.
    코드로 사후 제거하면 누구의 칭호를 버릴지 정해야 하는데 그 기준이 없다.

    temperature 0.9 ― 대사가 재미있어야 한다. 창작이므로 0 이 아니고,
    그래서 셀 5 가 여러 번 돈다.
    """
    user = USER_TT.format(
        winner=res["winner"],
        players=render_players_TT(res),
        votes=VOTES_TT,
        public_day_utterances=render_utterances_TT(utterances),
    )
    if show_prompt:
        print("=== SYSTEM ===\n" + SYSTEM_TT)
        print("\n=== USER ===\n" + user + "\n")

    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,      # gpt-4o-mini 는 max_tokens.
                                    # gpt-5.x 로 바꾸면 max_completion_tokens 로
        messages=[{"role": "system", "content": SYSTEM_TT},
                  {"role": "user",   "content": user}],
    )
    info = {"sec": time.perf_counter() - t0,
            "in":  resp.usage.prompt_tokens,
            "out": resp.usage.completion_tokens}

    raw = resp.choices[0].message.content
    try:
        parsed = json.loads(strip_fence(raw))
    except json.JSONDecodeError:
        # 파싱 실패도 결과로 본다. 실측된 실패 원인은 하나였다 ―
        # reason 안에서 발언을 큰따옴표로 감싸 JSON 문자열이 끊겼다.
        # 반복되면 OUTPUT_TT 에 "인용은 작은따옴표만" 한 줄을 넣는다.
        return raw, None, ["JSON 파싱 실패"], info

    return raw, parsed, check_titles_TT(parsed, res["players"]), info


print("칭호 대상", len(RES_TT["players"]), "명 ·",
      "플레이어 정보", len(render_players_TT(RES_TT)), "자 ·",
      "낮 대화", len(render_utterances_TT(DAY_TT)), "자")


칭호 대상 6 명 · 플레이어 정보 285 자 · 낮 대화 3817 자


In [58]:
# ===== 셀 5 / 5 : 실행 =====

RUNS_TT = 1     # 1회로 판단하지 않는다. temperature 0.9 라 회차마다 달라진다.
                # 프롬프트 다듬는 동안은 1회, 문안 확정 뒤 마지막에 3회로 본다

seen_TT = {}    # 닉네임 → [회차별 칭호]. 흔들림을 마지막에 한 번에 본다

for i in range(1, RUNS_TT + 1):
    raw, parsed, problems, info = award_titles_TT(RES_TT, DAY_TT)

    print("=" * 70)
    print(f"[{i}회차] {info['sec']:.1f}초 · 입력 {info['in']} · 출력 {info['out']}토큰")
    for p in problems:
        print("   ⚠️", p)
    print("=" * 70)

    if parsed is None:
        print(raw)
        continue

    for t in parsed.get("titles", []):
        nick = t.get("nickname")
        role = RES_TT["roles"].get(nick, "?")
        won = "승" if role == RES_TT["winner"] else "패"
        alive = "생존" if nick in RES_TT["survivors"] else "사망"
        print(f"🏆 {t.get('title')} — {nick} ({role}/{won}/{alive})")
        print(f"   “{t.get('quote')}”")
        print(f"   근거: {t.get('reason')}")      # 검증용. 실제 API 에서는 뺀다
        print()
        seen_TT.setdefault(nick, []).append(t.get("title"))

if RUNS_TT > 1:
    print("=" * 70)
    print("회차별 칭호 비교 — 매번 나오는 것이 그 사람의 실제 특징이다")
    print("=" * 70)
    for nick in RES_TT["players"]:
        print(f"{nick:<5} {' | '.join(seen_TT.get(nick, []))}")


[1회차] 6.0초 · 입력 4077 · 출력 461토큰
🏆 침착한 여론 장악자 — 라면왕 (시민/승/생존)
   “모든 상황을 묘사하며 조심스럽게 의심을 이끌어내셨습니다.”
   근거: 초코비를 의심하며 대화를 주도했고, 마지막 라운드에서 마피아를 정확히 지목하여 승리에 기여했다.

🏆 억울한 시민상 — 밤하늘 (시민/승/사망)
   “정말 의심받을 만한 행동이 없었는데 최후의 순간에 희생되셨군요.”
   근거: 라운드 2에서 사망했지만 특별한 의심을 받지 않았으며, 상황 정리에 참여했다.

🏆 완벽한 변명왕 — 초코비 (마피아/패/사망)
   “상황에 따라 변하는 발언이 인상 깊었습니다!”
   근거: 주요 의심을 받았음에도 불구하고 방어적인 자세로 상황을 반전하려 했지만 결국 패배했다.

🏆 무고한 희생양 — 소금빵 (시민/승/사망)
   “첫 라운드에 마피아의 희생물이 되어버린 안타까운 상황이었습니다.”
   근거: 첫 라운드에 마피아에게 살해당해 시민으로서 힘을 내지 못했다.

🏆 믿음직한 시민상 — 곰돌이 (시민/승/생존)
   “상황을 끊임없이 정리하며 믿음을 주는 플레이를 보여주셨습니다!”
   근거: 라운드 내내 적극적으로 의사소통하고 시청자들 사이에서 신뢰를 쌓았다.

🏆 자기 방어의 달인 — 딸기우유 (마피아/패/사망)
   “열심히 자신의 무죄를 주장하시느라 수고 많으셨습니다.”
   근거: 최후변론에서 자신을 방어하려 했지만 의심을 사면서 결국 처형되었다.



======================================================================
[1회차] 6.0초 · 입력 4077 · 출력 461토큰
======================================================================
🏆 침착한 여론 장악자 — 라면왕 (시민/승/생존)
   “모든 상황을 묘사하며 조심스럽게 의심을 이끌어내셨습니다.”
   근거: 초코비를 의심하며 대화를 주도했고, 마지막 라운드에서 마피아를 정확히 지목하여 승리에 기여했다.

🏆 억울한 시민상 — 밤하늘 (시민/승/사망)
   “정말 의심받을 만한 행동이 없었는데 최후의 순간에 희생되셨군요.”
   근거: 라운드 2에서 사망했지만 특별한 의심을 받지 않았으며, 상황 정리에 참여했다.

🏆 완벽한 변명왕 — 초코비 (마피아/패/사망)
   “상황에 따라 변하는 발언이 인상 깊었습니다!”
   근거: 주요 의심을 받았음에도 불구하고 방어적인 자세로 상황을 반전하려 했지만 결국 패배했다.

🏆 무고한 희생양 — 소금빵 (시민/승/사망)
   “첫 라운드에 마피아의 희생물이 되어버린 안타까운 상황이었습니다.”
   근거: 첫 라운드에 마피아에게 살해당해 시민으로서 힘을 내지 못했다.

🏆 믿음직한 시민상 — 곰돌이 (시민/승/생존)
   “상황을 끊임없이 정리하며 믿음을 주는 플레이를 보여주셨습니다!”
   근거: 라운드 내내 적극적으로 의사소통하고 시청자들 사이에서 신뢰를 쌓았다.

🏆 자기 방어의 달인 — 딸기우유 (마피아/패/사망)
   “열심히 자신의 무죄를 주장하시느라 수고 많으셨습니다.”
   근거: 최후변론에서 자신을 방어하려 했지만 의심을 사면서 결국 처형되었다.